***

Preparing Workspace

***

In [ ]:
# pip install kaleido==0.1.0.post1 # Remember to use this version of kaleido, otherwise you will wait indefinitely for an image to write

In [ ]:

## Importing packages ----

import numpy as np
import pandas as pd
import geopandas as gpd
import getpass
from pathlib import Path
import os
from tqdm import tqdm
import re
from datetime import date
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
import yaml
from IPython.display import display

import openpyxl
from openpyxl.drawing.image import Image
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Font
from openpyxl.styles import numbers
from openpyxl.styles import Border, Side
border_thin = Side(style='thin')

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.offline import plot
import plotly.subplots as sp
from plotly.subplots import make_subplots
pd.options.display.float_format = '{:.2f}'.format


## Setting file paths ---

user = getpass.getuser()
path_users = Path.home()

path_sp   = path_users / 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents'
path_prod = path_sp    / 'Products' / 'RHNA'
path_raw  = path_prod  / 'New Data Collected'

path_git = path_users / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
path_census = path_git  / 'Data' / 'Census'
path_config0 = path_git  / 'config'
path_config  = path_census / 'config'

path_prod = path_git / 'Products' / 'RHNA'
path_data = path_prod / 'Data'
path_yaml = path_data / 'RHNA_indicators.yaml'
path_func = path_data / 'RHNA_functions.py'
path_py = path_prod / 'python'

# path_i = Path('I:\Projects\Josh\RHNA')
# path_out = path_i / 'Final Products'
path_out = Path(r'C:\Users\jfontes\Documents\Projects\General\RHNA\Final Products')

## User defined ---

export=False
list_indicators = []

with open(path_yaml, 'r') as yaml_file:
    dict_about = yaml.load(yaml_file, Loader=yaml.SafeLoader)


***

Testing

***

In [ ]:


indicator = 'RHNA_DISAB_1'


# Set indicator
source = 'ACS5'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Population'


## Importing ---

df_places, df_counties, df_mpo = import_rhna(path_raw, indicator)



## Organizing ---

df_places['NAME'] = df_places['NAME'].str.replace(' CDP, California' , '', regex=True)
df_places['NAME'] = df_places['NAME'].str.replace(' city, California', '', regex=True)
df_places = df_places.rename(columns={'NAME':'Geography'})
df_places = df_places[df_places['Year'] == df_places['Year'].max()]
df_places = df_places.reset_index(drop=True)


df_places['Category'] = df_places['Variable'].copy()
df_places['Category'] = df_places['Category'].str.replace('With a ', '')
df_places['Category'] = df_places['Category'].str.replace('With an ', '')
df_places['Category'] = df_places['Category'].str.replace('No ', '')
df_places['Category'] = df_places['Category'].str.replace(' difficulty', '')

df_places['Percentage'] = df_places['Population'] / df_places.groupby(['County Name', 'Geography', 'Category'])['Population'].transform('sum')

df_places = df_places[~df_places['Variable'].str.contains('No')]
df_places = df_places[['County Name', 'Geography', 'Variable',  values, 'Percentage']].drop_duplicates()


df_places['Sort'] = pd.Categorical(df_places['Variable'], ['With an ambulatory difficulty', 'With an independent living difficulty', 'With a hearing difficulty'
                                                           , 'With a self-care difficulty', 'With a cognitive difficulty', 'With a vision difficulty'])
df_places = df_places.sort_values(['County Name', 'Geography', 'Sort'], ascending=[True, True, True])
df_places = df_places.drop(['Sort'], axis = 1)
df_places = df_places.reset_index(drop=True)

counties = list(df_places['County Name'].unique())


for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_places_sub = df_places.copy()
    df_places_sub = df_places_sub[df_places_sub['County Name'] == county]
    jurisdictions = df_places_sub['Geography'].unique()
    
    for jurisdiction in tqdm(jurisdictions):

        tqdm.write(jurisdiction)

        df_prod = df_places_sub[df_places_sub['Geography'] == jurisdiction]
        df_prod = df_prod[['Variable', values, 'Percentage']].drop_duplicates()
        df_plot = df_prod.copy()
        df_plot['Percentage'] = round(df_plot['Percentage'] * 100, 1)

        ## Plotting ---

        fig = px.bar(df_plot, x='Variable', y='Percentage')
        fig.update_traces(marker_color='#1E90FF')
        fig.update_traces(hovertemplate="%{y}")
        fig.update_yaxes(ticksuffix='%')

        path_plots = path_out / county.replace(' County', '') / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
        
    
        ## Exporting ---
        
        if export:
            export_rhna(df_prod)

# list_indicators.append(indicator)



In [ ]:



# indicator = 'RHNA_OVER_6'


# # Set indicator
# source = 'ACS5'
# with path_func.open("r") as f: exec(f.read())
# title = dict_about[source][indicator.replace('RHNA_', '')]['Indicator Title'][0]
# values = 'Households'
# columns = 'Burden'


# ## Importing ---

# df_places = pd.read_excel(os.path.join(path_raw, f'{indicator} Places ACS5.xlsx'))


# ## Organizing ---

# def re_remove_post(x, exp = ':'):
#     if x == 'nan':
#         return 'nan'
#     else:
#         return x.split(exp, 1)[0]

# def re_remove_pre(x, exp = ':  '):
#     if x == 'nan':
#         return 'nan'
#     else:
#         return str(x.split(exp, 1)[1])

# df_places['NAME'] = df_places['NAME'].str.replace(' CDP, California' , '', regex=True)
# df_places['NAME'] = df_places['NAME'].str.replace(' city, California', '', regex=True)
# df_places = df_places.rename(columns={'NAME':'Geography'})
# df_places = df_places[df_places['Year'] == df_places['Year'].max()]
# df_places = df_places.reset_index(drop=True)
# df_places['Tenure'  ] = df_places['Variable'].apply(re_remove_post)
# df_places['Burden'  ] = df_places['Variable'].apply(re_remove_pre )
# df_places['Variable'] = df_places['Tenure'].copy()
# df_places['Percentage'] = df_places['Households'] / df_places.groupby(['County Name', 'Geography', 'Variable'])['Households'].transform('sum')

# df_places = df_places[['County Name', 'Geography', values, 'Variable', columns, 'Percentage']]

# df_places['Sort'] = pd.Categorical(df_places['Burden'], ['0%-30% of income used for housing', '30%-50% of income used for housing', '50% or more of income used for housing', 'Not computed'])
# df_places = df_places.sort_values(['County Name', 'Geography', 'Variable', 'Sort'], ascending=[True, True, True, True])
# df_places = df_places.drop(['Sort'], axis = 1)
# df_places = df_places.reset_index(drop=True)
# # df_places


# counties = list(df_places['County Name'].unique())


# for county in counties:
    
#     print();print()
#     print(county)
#     time.sleep(2)

#     df_places_sub = df_places.copy()
#     df_places_sub = df_places_sub[df_places_sub['County Name'] == county]
#     jurisdictions = df_places_sub['Geography'].unique()
    
#     for jurisdiction in tqdm(jurisdictions, position=0):

#         tqdm.write(jurisdiction)

#         df_prod, df_pct = pivot_rhna(indicator, df_places_sub, county, jurisdiction, columns, values)

#         ## Plotting ---

#         df_plot = df_places_sub[df_places_sub['Geography'] == jurisdiction]
#         df_plot['Percentage'] = round(df_plot['Percentage']*100, 1)
        
#         color_map = {
#                 "0%-30% of income used for housing":"#1F45FC",
#                 "30%-50% of income used for housing":"#1E90FF",
#                 "50% or more of income used for housing":"#9DC209",
#                 "Not computed":"#FBB117"
#         }

#         fig = px.bar(df_plot, x='Variable', y='Percentage'
#                      , color = columns
#                      , color_discrete_map=color_map)
        
#         fig.update_yaxes(dtick=10, ticksuffix='%', range = [0,102])
#         fig.update_layout(legend={'traceorder': 'reversed'})
#         fig.update_traces(hovertemplate="%{y}")
            
#         path_plots = path_out / county.replace(' County', '') / jurisdiction / 'Supplemental'
#         plot_rhna(export=export)
    
#         ## Exporting ---
#         if export:
#             export_rhna(df_prod, df_pct)

# list_indicators.append(indicator)



***

Indicators

***

In [ ]:


## Census Bureau ---

## ACS 
# POPEMP-02, POPEMP-03, POPEMP-04, POPEMP-05, POPEMP-06, POPEMP-07, POPEMP-08, POPEMP-09, POPEMP-10, 
# POPEMP-16, POPEMP-17, POPEMP-18, POPEMP-19, POPEMP-20, POPEMP-22, POPEMP-23, POPEMP-24, POPEMP-25, 
# HSG-02, HSG-03, HSG-04, HSG-05, HSG-06, HSG-07, HSG-09, HSG-10, 
# OVER-03, OVER-06, OVER-07, LGFEM-01, LGFEM-02, LGFEM-04, LGFEM-05, 
# SEN-02, SEN-04, DISAB-01, DISAB-02, DISAB-03, HOMELS-02, HOMELS-03, ELI-03, AFFH-02, AFFH-03

## DEC
# POPEMP-02, POPEMP-04, POPEMP-17

## LEHD
# POPEMP-11, POPEMP-12, POPEMP-13, POPEMP-14


## HUD ---

## CHAS
# POPEMP-21, OVER-01, OVER-02, OVER-04, OVER-05, OVER-08, OVER-09, LGFEM-03, SEN-01, SEN-03, ELI-01, ELI-02

## CoC
# HOMELS-01, HOMELS-02, HOMELS-03, HOMELS-04


## DOF ---

# POPEMP-01, HSG-01


## EDU ---

# FARM-01

## Zillow ---

# HSG-08



***

POPEMP

***

In [ ]:


indicator_name = 'RHNA_POPEMP_1'


# Set indicator
source = 'DOF'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
base_year = 2001

# DOF E5 Population time-series, with growth rate relative to 1990
# Compare Jurisdictions to County and MPO


## Importing ---

df_places, df_counties, df_mpo = import_rhna(path_raw, indicator_name)


## Organizing ---

df_places, df_counties, df_mpo = clean_rhna(df_places, df_counties, df_mpo)

counties = df_counties['Geography'].unique()

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_places_sub, df_counties_sub = sub_rhna(df_places, df_counties, county)
    jurisdictions = df_places_sub['Geography'].unique()
    
    for jurisdiction in tqdm(jurisdictions):
                        
        ## Plotting ---

        df_plot1, df_plot2, df_plot3 = index_rhna(df_places_sub, df_counties_sub, df_mpo, jurisdiction)
        df_prod = pivot_rhna(indicator_name, df_plot1, county, jurisdiction, df_plot2, df_plot3)

        df_plot2['Geography'] = df_plot2['Geography'] + ' County'
        df_plot = pd.concat([df_plot1, df_plot2, df_plot3])
        df_plot = df_plot.drop(['County', 'Population'], axis=1).rename(columns = {'growth': 'Percent Difference'})
            
        color_map  = {
            'SACOG': '#9DC209'
            , f'{county} County': '#1E90FF'
            , jurisdiction: '#FBB117'
        }
    
        fig = px.line(df_plot, x='Year', y='Percent Difference', markers=True
                         , color = 'Geography'
                         , color_discrete_map=color_map)
        
        # title = f'<b>{plot_title}</b>'

        range_min = df_plot['Percent Difference'].min()-4
        range_max = df_plot['Percent Difference'].max()+4
        range_diff = abs(range_max-range_min)
        if range_diff <= 10:
            dtick = 1
        elif (range_diff > 10) & (range_diff <= 50):
            dtick = 5
        elif (range_diff > 50) & (range_diff <= 100):
            dtick = 10
        elif (range_diff > 100) & (range_diff <= 200):
            dtick = 25
        else:
            dtick = 50
        fig.update_yaxes(ticksuffix='%', dtick=dtick, range = [range_min, range_max])
        year_min = df_plot['Year'].min()-0.5
        year_max = df_plot['Year'].max()+0.5
        fig.update_xaxes(dtick=1, range = [year_min, year_max])
        fig.update_layout(legend={'traceorder': 'reversed'})
        fig.update_traces(hovertemplate="%{y}")
    
        path_plots = path_out / county / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod)
            

list_indicators.append(indicator_name)



In [ ]:


# # Set indicator
# source = 'DEC'
# indicator_name = 'RHNA_POPEMP_2'
# title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
# values = 'Population'
# columns = 'Race_Ethnicity'
# # export = False

# # Compare DEC 2000, 2010, 2020, with ACS5 2018-2023?
# # Jurisdiction only



In [ ]:


indicator_name = 'RHNA_POPEMP_3'


# Set indicator
source = 'ACS5'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Population'
columns = 'Race_Ethnicity'


## Importing ---

df_places, df_counties, df_mpo = import_rhna(path_raw, indicator_name)


## Organizing ---

df_places   = df_places  [df_places  ['Race_Ethnicity'] != 'All']
df_counties = df_counties[df_counties['Race_Ethnicity'] != 'All']
df_mpo      = df_mpo     [df_mpo     ['Race_Ethnicity'] != 'All']

df_places, df_counties, df_mpo = clean_rhna(df_places, df_counties, df_mpo, path_config0, columns, values)


counties = df_counties['Geography'].unique()

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_places_sub, df_counties_sub = sub_rhna(df_places, df_counties, county)
    jurisdictions = df_places_sub['Geography'].unique()
    
    for jurisdiction in tqdm(jurisdictions):
                
        df_prod, df_pct = pivot_rhna(indicator_name, df_places_sub, county, jurisdiction, columns, values, df_counties_sub, df_mpo)
        
        ## Plotting ---
        
        df_plot = pd.concat([df_places_sub[df_places_sub['Geography'] == jurisdiction], df_counties_sub, df_mpo])
        df_plot = df_plot.drop('Population', axis=1)
        df_plot['Percentage'] = round(df_plot['Percentage'], 1)
        df_plot['Sort'] = pd.Categorical(df_plot['Geography'], [jurisdiction, county, 'SACOG'])
        df_plot['Sort_eth'] = pd.Categorical(df_plot['Race_Ethnicity'], [
            'American Indian or Alaska Native (NH)'
            , 'Native Hawaiian or other Pacific Islander (NH)'
            , 'Some other race (NH)'
            , 'Two or more races (NH)'
            , 'Black or African American (NH)'
            , 'Asian (NH)'
            , 'Hispanic or Latino'
            , 'White (NH)'
        ])
        df_plot = df_plot.sort_values(['Sort', 'Sort_eth'], ascending=[True, False])
        df_plot = df_plot.drop(['Sort', 'Sort_eth'], axis=1)
            
        color_map  = {
            'American Indian or Alaska Native (NH)': '#A97142'
            , 'Native Hawaiian or other Pacific Islander (NH)': '#006A4E'
            , 'Some other race (NH)': '#7E587E'
            , 'Two or more races (NH)': '#1F45FC'
            , 'Asian (NH)': '#9DC209'
            , 'Black or African American (NH)': '#1E90FF'
            , 'Hispanic or Latino': '#FBB117'
            , 'White (NH)': '#DC381F'
        }
    
        fig = px.bar(df_plot, x='Geography', y='Percentage'
                     , color = columns
                     , color_discrete_map=color_map)
        
        # title = f'<b>{plot_title}</b>'
        fig.update_yaxes(dtick=10, ticksuffix='%', range = [0,102])
        fig.update_layout(legend={'traceorder': 'reversed'})
        fig.update_traces(hovertemplate="%{y}")
    
        path_plots = path_out / county.replace(' County', '') / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            

list_indicators.append(indicator_name)



In [ ]:

# # Set indicator
# source = 'ACS5'
# indicator_name = 'RHNA_POPEMP_4 '
# title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
# values = 'Population'
# columns = 'Race_Ethnicity'
# # export = False

# # Compare DEC 2000, 2010, 2020, with ACS5 2018-2023?
# # Jurisdiction only



In [ ]:


indicator_name = 'RHNA_POPEMP_5'


# Set indicator
source = 'ACS5'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Population'
columns = 'Variable'


## Importing ---

df_places, df_counties, df_mpo = import_rhna(path_raw, indicator_name)


## Organizing ---

df_places, df_counties, df_mpo = clean_rhna(df_places, df_counties, df_mpo, path_config0, columns, values)


counties = df_counties['Geography'].unique()

for county in counties:

    print();print()
    print(county)
    time.sleep(2)

    df_places_sub, df_counties_sub = sub_rhna(df_places, df_counties, county)
    jurisdictions = df_places_sub['Geography'].unique()
    
    for jurisdiction in tqdm(jurisdictions):
                
        df_prod, df_pct = pivot_rhna(indicator_name, df_places_sub, county, jurisdiction, columns, values, df_counties_sub, df_mpo)
    
    
        ## Plotting ---
        
        df_plot = pd.concat([df_places_sub[df_places_sub['Geography'] == jurisdiction], df_counties_sub, df_mpo])
        df_plot = df_plot.drop(values, axis=1)
        df_plot['Percentage'] = round(df_plot['Percentage'], 1)
            
        color_map  = {
            'Same house': '#1E90FF'
            , 'Same city or town': '#1F45FC'
            , 'Same county': '#9DC209'
            , 'Elsewhere in CA': '#7E587E'
            , 'Elsewhere in U.S.': '#FBB117'
            , 'Abroad': '#DC381F'
        }
    
        fig = px.bar(df_plot, x='Geography', y='Percentage'
                     , color = columns
                     , color_discrete_map=color_map)
        
        fig.update_yaxes(dtick=10, ticksuffix='%', range = [0,102])
        fig.update_layout(legend={'traceorder': 'reversed'})
        fig.update_traces(hovertemplate="%{y}")
    
        path_plots = path_out / county.replace(' County', '') / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            

list_indicators.append(indicator_name)



In [ ]:


indicator_name = 'RHNA_POPEMP_6'


# Set indicator
source = 'ACS5'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Population'
columns = 'Variable'


## Importing ---

df_places, df_counties, df_mpo = import_rhna(path_raw, indicator_name)


## Organizing ---

df_places, df_counties, df_mpo = clean_rhna(df_places, df_counties, df_mpo, path_config0, columns, values)


counties = df_counties['Geography'].unique()

for county in counties:

    print();print()
    print(county)
    time.sleep(2)
    
    df_places_sub, df_counties_sub = sub_rhna(df_places, df_counties, county)
    jurisdictions = df_places_sub['Geography'].unique()
    
    for jurisdiction in tqdm(jurisdictions):
                
        df_prod, df_pct = pivot_rhna(indicator_name, df_places_sub, county, jurisdiction, columns, values, df_counties_sub, df_mpo)
    
    
        ## Plotting ---
        
        df_plot = pd.concat([df_places_sub[df_places_sub['Geography'] == jurisdiction], df_counties_sub, df_mpo])
        df_plot = df_plot.drop(values, axis=1)
        df_plot['Percentage'] = round(df_plot['Percentage'], 1)
            
        color_map  = {
            'Agriculture & Natural Resources': '#1E90FF'
            , 'Construction': '#1F45FC'
            , 'Manufacturing, Wholesale, & Transportation': '#9DC209'
            , 'Retail': '#7FFFD4'
            , 'Information': '#7E587E'
            , 'Finance & Professional Services': '#FBB117'
            , 'Health & Educational Services': '#008000'
            , 'Other': '#DC381F'
        }
    
        fig = px.bar(df_plot, x='Geography', y='Percentage'
                     , color = columns
                     , color_discrete_map=color_map)
        
        fig.update_yaxes(dtick=10, ticksuffix='%', range = [0,102])
        fig.update_layout(legend={'traceorder': 'reversed'})
        fig.update_traces(hovertemplate="%{y}")
    
        path_plots = path_out / county.replace(' County', '') / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            

list_indicators.append(indicator_name)



In [ ]:


indicator_name = 'RHNA_POPEMP_7'


# Set indicator
source = 'ACS5'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Population'
columns = 'Variable'


## Importing ---

df_places, df_counties, df_mpo = import_rhna(path_raw, indicator_name)


## Organizing ---

df_places, df_counties, df_mpo = clean_rhna(df_places, df_counties, df_mpo, path_config0, columns, values)


counties = df_counties['Geography'].unique()

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_places_sub, df_counties_sub = sub_rhna(df_places, df_counties, county)
    jurisdictions = df_places_sub['Geography'].unique()
    
    for jurisdiction in tqdm(jurisdictions):
                
        df_prod, df_pct = pivot_rhna(indicator_name, df_places_sub, county, jurisdiction, columns, values, df_counties_sub, df_mpo)
    
    
        ## Plotting ---
        
        df_plot = pd.concat([df_places_sub[df_places_sub['Geography'] == jurisdiction], df_counties_sub, df_mpo])
        df_plot = df_plot.drop(values, axis=1)
        df_plot['Percentage'] = round(df_plot['Percentage'], 1)
            
        color_map  = {
            'Management, Business, Science, and Arts occupations': '#1F45FC'
            , 'Service occupations': '#1E90FF'
            , 'Sales and Office occupations': '#9DC209'
            , 'Natural Resources, Construction, and Maintenance occupations': '#FBB117'
            , 'Production, Transportation, and Material Moving occupations': '#7E587E'
        }
    
        fig = px.bar(df_plot, x='Geography', y='Percentage'
                     , color = columns
                     , color_discrete_map=color_map)
        
        fig.update_yaxes(dtick=10, ticksuffix='%', range = [0,102])
        fig.update_layout(legend={'traceorder': 'reversed'})
        fig.update_traces(hovertemplate="%{y}")
    
        path_plots = path_out / county.replace(' County', '') / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            

list_indicators.append(indicator_name)



In [ ]:


indicator_name = 'RHNA_POPEMP_8'


# Set indicator
source = 'ACS5'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Population'
columns = 'Variable'


## Importing ---

df_places, df_counties, df_mpo = import_rhna(path_raw, indicator_name)


## Organizing ---

df_places, df_counties, df_mpo = clean_rhna(df_places, df_counties, df_mpo, path_config0, columns, values)


counties = df_counties['Geography'].unique()

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_places_sub, df_counties_sub = sub_rhna(df_places, df_counties, county)
    jurisdictions = df_places_sub['Geography'].unique()
    
    for jurisdiction in tqdm(jurisdictions):
                
        df_prod, df_pct = pivot_rhna(indicator_name, df_places_sub, county, jurisdiction, columns, values, df_counties_sub, df_mpo)
    
    
        ## Plotting ---
        
        df_plot = pd.concat([df_places_sub[df_places_sub['Geography'] == jurisdiction], df_counties_sub, df_mpo])
        df_plot = df_plot.drop(values, axis=1)
        df_plot['Percentage'] = round(df_plot['Percentage'], 1)
            
        color_map  = {
            'Private company workers': '#1F45FC'
            , 'Self-employed workers': '#7E587E'
            , 'Private not-for-profit workers': '#1E90FF'
            , 'Local and state government workers': '#9DC209'
            , 'Federal government workers': '#FBB117'
            , 'Unpaid family workers': '#7FFFD4'
        }
    
        fig = px.bar(df_plot, x='Geography', y='Percentage'
                     , color = columns
                     , color_discrete_map=color_map)
        
        fig.update_yaxes(dtick=10, ticksuffix='%', range = [0,102])
        fig.update_layout(legend={'traceorder': 'reversed'})
        fig.update_traces(hovertemplate="%{y}")
    
        path_plots = path_out / county.replace(' County', '') / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            

list_indicators.append(indicator_name)



In [ ]:


indicator_name = 'RHNA_POPEMP_9'


# Set indicator
source = 'ACS5'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Population'
columns = 'Variable'


## Importing ---

df_places, df_counties, df_mpo = import_rhna(path_raw, indicator_name)


## Organizing ---

df_places, df_counties, df_mpo = clean_rhna(df_places, df_counties, df_mpo, path_config0, columns, values)


counties = df_counties['Geography'].unique()

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_places_sub, df_counties_sub = sub_rhna(df_places, df_counties, county)
    jurisdictions = df_places_sub['Geography'].unique()
    
    for jurisdiction in tqdm(jurisdictions):
                
        df_prod, df_pct = pivot_rhna(indicator_name, df_places_sub, county, jurisdiction, columns, values, df_counties_sub, df_mpo)
    
    
        ## Plotting ---
        
        df_plot = pd.concat([df_places_sub[df_places_sub['Geography'] == jurisdiction], df_counties_sub, df_mpo])
        df_plot = df_plot.drop(values, axis=1)
        df_plot['Percentage'] = round(df_plot['Percentage'], 1)
            
        color_map  = {
            'Private company workers': '#1F45FC'
            , 'Self-employed workers': '#7E587E'
            , 'Private not-for-profit workers': '#1E90FF'
            , 'Local and state government workers': '#9DC209'
            , 'Federal government workers': '#FBB117'
            , 'Unpaid family workers': '#7FFFD4'
        }
    
        fig = px.bar(df_plot, x='Geography', y='Percentage'
                     , color=columns
                     , color_discrete_map=color_map)
        
        fig.update_yaxes(dtick=10, ticksuffix='%', range = [0,102])
        fig.update_layout(legend={'traceorder': 'reversed'})
        fig.update_traces(hovertemplate="%{y}")
    
        path_plots = path_out / county.replace(' County', '') / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            

list_indicators.append(indicator_name)



In [ ]:


indicator_name = 'RHNA_POPEMP_10'


# Set indicator
source = 'ACS5'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Population'
columns = 'Category'


## Importing ---

df_places_a = pd.read_excel(os.path.join(path_raw, f'{indicator_name}a Places ACS5.xlsx'))
df_places_b = pd.read_excel(os.path.join(path_raw, f'{indicator_name}b Places ACS5.xlsx'))
df_places = pd.concat([df_places_a, df_places_b])


## Organizing ---

def re_remove_post(x, exp = ':'):
    if x == 'nan':
        return 'nan'
    else:
        return x.split(exp, 1)[0]

def re_remove_pre(x, exp = ':  '):
    if x == 'nan':
        return 'nan'
    else:
        return str(x.split(exp, 1)[1])

df_places['NAME'] = df_places['NAME'].str.replace(' CDP, California' , '', regex=True)
df_places['NAME'] = df_places['NAME'].str.replace(' city, California', '', regex=True)
df_places = df_places.rename(columns={'NAME':'Geography'})
df_places = df_places[df_places['Year'] == df_places['Year'].max()]
df_places = df_places.reset_index(drop=True)
df_places['Category'] = df_places['Variable'].apply(re_remove_post)
df_places['Variable'] = df_places['Variable'].apply(re_remove_pre )

df_places = df_places[['County Name', 'Geography', values, columns, 'Variable', 'Percentage']]

df_places['Sort'] = pd.Categorical(df_places['Variable'], ['75k or more', '50k to 75k', '25k to 50k', '10k to 25k', 'Less than 10k'])
df_places = df_places.sort_values(['County Name', 'Geography', 'Category', 'Sort'], ascending=[True, True, True, False])
df_places = df_places.drop(['Sort'], axis = 1)

counties = list(df_places['County Name'].unique())


for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_places_sub = df_places.copy()
    df_places_sub = df_places_sub[df_places_sub['County Name'] == county]
    jurisdictions = df_places_sub['Geography'].unique()
    
    for jurisdiction in tqdm(jurisdictions):

        df_prod, df_pct = pivot_rhna(indicator_name, df_places_sub, county, jurisdiction, columns, values)

        ## Plotting ---

        df_plot = df_places_sub[df_places_sub['Geography'] == jurisdiction]
        
        color_map  = {
            'Place of residence': '#9DC209'
            , 'Place of work': '#1F45FC'
        }

        fig = px.bar(df_plot, x='Variable', y=values
                     , color = columns
                     , barmode='group'
                     , color_discrete_map=color_map)
        
        fig.update_traces(hovertemplate="%{y}")
    
        path_plots = path_out / county.replace(' County', '') / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            

list_indicators.append(indicator_name)



In [ ]:


indicator_name = 'RHNA_POPEMP_16'


# Set indicator
source = 'ACS5'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Population'
columns = 'Variable'



## Importing ---

df_places, df_counties, df_mpo = import_rhna(path_raw, indicator_name)


## Organizing ---

df_places, df_counties, df_mpo = clean_rhna(df_places, df_counties, df_mpo, path_config0, columns, values)


counties = df_counties['Geography'].unique()


for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_places_sub, df_counties_sub = sub_rhna(df_places, df_counties, county)
    jurisdictions = df_places_sub['Geography'].unique()
    
    for jurisdiction in tqdm(jurisdictions):
                
        df_prod, df_pct = pivot_rhna(indicator_name, df_places_sub, county, jurisdiction, columns, values, df_counties_sub, df_mpo)
        
    
        ## Plotting ---
        
        df_plot = pd.concat([df_places_sub[df_places_sub['Geography'] == jurisdiction], df_counties_sub, df_mpo])
        df_plot = df_plot.drop(values, axis=1)
        df_plot['Percentage'] = round(df_plot['Percentage'], 1)
        
        color_map = {
                 "Owner occupied":"#1F45FC",
                 "Renter occupied": "#9DC209",
        }
        
        fig = px.bar(df_plot, x='Geography', y='Percentage'
                     , color = columns
                     , color_discrete_map=color_map)
        
        fig.update_yaxes(dtick=10, ticksuffix='%', range = [0,102])
        fig.update_layout(legend={'traceorder': 'reversed'})
        fig.update_traces(hovertemplate="%{y}")
    
        path_plots = path_out / county.replace(' County', '') / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            
    
list_indicators.append(indicator_name)



In [ ]:


indicator_name = 'RHNA_POPEMP_19'


# Set indicator
source = 'ACS5'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Households'
columns = 'Category'


## Importing ---

df_places_a = pd.read_excel(os.path.join(path_raw, f'{indicator_name}a Places ACS5.xlsx'))
df_places_b = pd.read_excel(os.path.join(path_raw, f'{indicator_name}b Places ACS5.xlsx'))
df_places = pd.concat([df_places_a, df_places_b])


## Organizing ---

def re_remove_post(x, exp = ':'):
    if x == 'nan':
        return 'nan'
    else:
        return x.split(exp, 1)[0]

def re_remove_pre(x, exp = ': '):
    if x == 'nan':
        return 'nan'
    else:
        return str(x.split(exp, 1)[1])

df_places['NAME'] = df_places['NAME'].str.replace(' CDP, California' , '', regex=True)
df_places['NAME'] = df_places['NAME'].str.replace(' city, California', '', regex=True)
df_places = df_places.rename(columns={'NAME':'Geography'})
df_places = df_places[df_places['Year'] == df_places['Year'].max()]
df_places = df_places.reset_index(drop=True)
df_places['Category'] = df_places['Variable'].apply(re_remove_post)
df_places['Variable'] = df_places['Variable'].apply(re_remove_pre )

df_places = df_places[['County Name', 'Geography', values, columns, 'Variable', 'Percentage']]

df_places['Sort'] = pd.Categorical(df_places['Variable'], ['Moved in 2021 or later', 'Moved in 2018 to 2020', 'Moved in 2010 to 2017'
                                                           , 'Moved in 2000 to 2009', 'Moved in 1999 or earlier'])
df_places = df_places.sort_values(['County Name', 'Geography', 'Category', 'Sort'], ascending=[True, True, True, False])
df_places = df_places.drop(['Sort'], axis = 1)

counties = list(df_places['County Name'].unique())


for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_places_sub = df_places.copy()
    df_places_sub = df_places_sub[df_places_sub['County Name'] == county]
    jurisdictions = df_places_sub['Geography'].unique()
    
    for jurisdiction in tqdm(jurisdictions):

        df_prod, df_pct = pivot_rhna(indicator_name, df_places_sub, county, jurisdiction, columns, values)

        ## Plotting ---

        df_plot = df_places_sub[df_places_sub['Geography'] == jurisdiction]
        
        color_map  = {
            'Owner occupied': '#9DC209'
            , 'Renter occupied': '#1F45FC'
        }

        fig = px.bar(df_plot, x='Variable', y=values
                     , color = columns
                     , barmode='group'
                     , color_discrete_map=color_map)
        
        fig.update_traces(hovertemplate="%{y}")
    
        path_plots = path_out / county.replace(' County', '') / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            

list_indicators.append(indicator_name)



In [ ]:


indicator_name = 'RHNA_POPEMP_21'


# Set indicator
source = 'CHAS'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Households'
columns = 'Tenure'


## Importing ---

file_chas = path_raw / f'HUD_CHAS_2017thru2021.csv'
df_chas = pd.read_csv(file_chas, dtype=str)


## Organizing ---

df_chas['Households'] = df_chas['Households'].astype(int)

estimates = ['T7_est3', 'T7_est24', 'T7_est45', 'T7_est66', 'T7_est87', 'T7_est109', 'T7_est130', 'T7_est151', 'T7_est172', 'T7_est193']

df_chas = df_chas[df_chas['Estimate'].isin(estimates)]
print(df_chas['Description 1'].unique())
print(df_chas['Description 2'].unique())
print(df_chas['Description 3'].unique())
print(df_chas['Description 4'].unique())


df_chas = df_chas[['County Name', 'name', 'Description 1', 'Description 2', 'Households']]
df_chas = df_chas.rename(columns = {'Description 1':'Tenure', 'Description 2':'Income Level'})
df_chas['name'] = df_chas['name'].str.replace(' city, California', '', regex=True)
df_chas['name'] = df_chas['name'].str.replace(' town, California', '', regex=True)

conditions = [
    df_chas['Income Level'] == 'household income is less than or equal to 30% of HAMFI'
    , df_chas['Income Level'] == 'household income is greater than 30% but less than or equal to 50% of HAMFI'
    , df_chas['Income Level'] == 'household income is greater than 50% but less than or equal to 80% of HAMFI'
    , df_chas['Income Level'] == 'household income is greater than 80% but less than or equal to 100% of HAMFI'
    , df_chas['Income Level'] == 'household income is greater than 100% of HAMFI'
]

choices = ['0%-30% of AMI', '31%-50% of AMI', '51%-80% of AMI', '81%-100% of AMI', 'Greater than 100% of AMI']

df_chas['Income Level'] = np.select(conditions, choices, default='no')
df_chas['Percentage'] = 100 * (df_chas['Households'] / df_chas.groupby(['County Name', 'name', 'Tenure'])['Households'].transform('sum'))
df_chas = df_chas.reset_index(drop=True)

display(df_chas)


counties = list(df_chas['County Name'].unique())

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_chas_sub = df_chas[df_chas['County Name'] == county]
    jurisdictions = df_chas_sub['name'].unique()
    
    for jurisdiction in tqdm(jurisdictions):

        df_prod = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        df_prod = df_prod.drop('Percentage', axis=1)
        df_prod = df_prod.pivot_table(index=['Income Level'], columns=columns, values=values).reset_index()

        df_pct = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        df_pct = df_pct.drop('Households', axis=1)
        df_pct = df_pct.pivot_table(index=['Income Level'], columns=columns, values='Percentage').reset_index()
        
        ## Plotting ---

        df_plot = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        
        color_map  = {
            'Owner occupied': '#9DC209'
            , 'Renter occupied': '#1F45FC'
        }

        fig = px.bar(df_plot, x='Income Level', y=values
                     , color = columns
                     , barmode='group'
                     , color_discrete_map=color_map)
        
        fig.update_traces(hovertemplate="%{y}")
    
        path_plots = path_out / county / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
        print(); print()
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            

list_indicators.append(indicator_name)


In [ ]:


indicator_name = 'RHNA_POPEMP_23'


# Set indicator
source = 'ACS5'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Population'
columns = 'Variable'


## Importing ---

df_places, df_counties, df_mpo = import_rhna(path_raw, indicator_name)


## Organizing ---

df_places, df_counties, df_mpo = clean_rhna(df_places, df_counties, df_mpo, path_config0, columns, values)


counties = df_counties['Geography'].unique()

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_places_sub, df_counties_sub = sub_rhna(df_places, df_counties, county)
    jurisdictions = df_places_sub['Geography'].unique()
    
    for jurisdiction in tqdm(jurisdictions):
                
        df_prod, df_pct = pivot_rhna(indicator_name, df_places_sub, county, jurisdiction, columns, values, df_counties_sub, df_mpo)
    
    
        ## Plotting ---
        
        df_plot = pd.concat([df_places_sub[df_places_sub['Geography'] == jurisdiction], df_counties_sub, df_mpo])
        df_plot = df_plot.drop(values, axis=1)
        df_plot['Percentage'] = round(df_plot['Percentage'], 1)
            
        color_map  = {
            'Female-headed family households': '#9DC209'
            , 'Male-headed family households': '#1E90FF'
            , 'Married-couple family households': '#1F45FC'
            , 'Other non-family households': '#7E587E'
            , 'Single-person households': '#FBB117'
        }
    
        fig = px.bar(df_plot, x='Geography', y='Percentage'
                     , color = columns
                     , color_discrete_map=color_map)
        
        fig.update_yaxes(dtick=10, ticksuffix='%', range = [0,102])
        fig.update_layout(legend={'traceorder': 'reversed'})
        fig.update_traces(hovertemplate="%{y}")
    
        path_plots = path_out / county.replace(' County', '') / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            

list_indicators.append(indicator_name)



In [ ]:


indicator_name = 'RHNA_POPEMP_24'


# Set indicator
source = 'ACS5'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Households'
columns = 'Variable'


## Importing ---

df_places, df_counties, df_mpo = import_rhna(path_raw, indicator_name)


## Organizing ---

df_places, df_counties, df_mpo = clean_rhna(df_places, df_counties, df_mpo, path_config0, columns, values)


counties = df_counties['Geography'].unique()

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_places_sub, df_counties_sub = sub_rhna(df_places, df_counties, county)
    jurisdictions = df_places_sub['Geography'].unique()
    
    for jurisdiction in tqdm(jurisdictions):
                
        df_prod, df_pct = pivot_rhna(indicator_name, df_places_sub, county, jurisdiction, columns, values, df_counties_sub, df_mpo)
    
    
        ## Plotting ---
        
        df_plot = pd.concat([df_places_sub[df_places_sub['Geography'] == jurisdiction], df_counties_sub, df_mpo])
        df_plot = df_plot.drop(values, axis=1)
        df_plot['Percentage'] = round(df_plot['Percentage'], 1)
            
        color_map  = {
            'Households with 1 or more children under 18': '#1E90FF'
            , 'Households with no children': '#9DC209'
        }
    
        fig = px.bar(df_plot, x='Geography', y='Percentage'
                     , color=columns
                     , color_discrete_map=color_map)
        
        fig.update_yaxes(dtick=10, ticksuffix='%', range = [0,102])
        fig.update_layout(legend={'traceorder': 'reversed'})
        fig.update_traces(hovertemplate="%{y}")
    
        path_plots = path_out / county.replace(' County', '') / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            

list_indicators.append(indicator_name)



***

HSG

***

In [ ]:


indicator_name = 'RHNA_HSG_1'


# Set indicator
source = 'DOF'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values='Housing Units'
columns='Year'


## Importing ---

file_in = path_raw / 'DOF_E5_and_E8_Jurisdictions.xlsx'
df_dof = pd.read_excel(file_in)


## Organizing ---

df_dof = df_dof[df_dof['MPO'] == 'SACOG']
df_dof = df_dof[df_dof['Year'].isin([2010, 2024])]
df_dof = df_dof.sort_values('Year')
df_dof['Year'] = df_dof['Year'].astype(str)

df_dof = df_dof[['County', 'Jurisdiction', 'Year', 'Single Attached', 'Single Detached', 'Two to Four', 'Five Plus', 'Mobile Homes']]

df_dof = df_dof.melt(id_vars=['County', 'Jurisdiction', 'Year'], var_name='Housing Type', value_name='Housing Units')
df_dof = df_dof.reset_index(drop=True)



counties = list(df_dof['County'].unique())

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_dof_sub = df_dof[df_dof['County'] == county]
    jurisdictions = df_dof_sub['Jurisdiction'].unique()

    for jurisdiction in tqdm(jurisdictions):

        print()
        df_prod = df_dof_sub[df_dof_sub['Jurisdiction'] == jurisdiction]
        df_prod['Year'] = 'Year ' + df_prod['Year']
        df_prod = df_prod.pivot_table(index=['Housing Type'], columns=columns, values=values).reset_index()
        display(df_prod); print()

        
        ## Plotting ---

        df_plot = df_dof_sub[df_dof_sub['Jurisdiction'] == jurisdiction]
        
        color_map = {
                 "2010":"#1F45FC",
                 "2024": "#9DC209",
        }

        fig = px.bar(df_plot, x='Housing Type', y=values
                     , color = columns
                     , barmode='group'
                     , color_discrete_map=color_map)
        
        fig.update_traces(hovertemplate="%{y}")
        # fig.update_layout(legend={'traceorder': 'reversed'})

        path_plots = path_out / county / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
        print(); print()
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod)
            

list_indicators.append(indicator_name)



In [ ]:


indicator_name = 'RHNA_HSG_2'


# Set indicator
source = 'ACS5'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Households'
columns = 'Variable'


## Importing ---

df_places, df_counties, df_mpo = import_rhna(path_raw, indicator_name)


## Organizing ---

df_places, df_counties, df_mpo = clean_rhna(df_places, df_counties, df_mpo, path_config0, columns, values)


counties = df_counties['Geography'].unique()

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_places_sub, df_counties_sub = sub_rhna(df_places, df_counties, county)
    jurisdictions = df_places_sub['Geography'].unique()
    
    for jurisdiction in tqdm(jurisdictions):
                
        df_prod, df_pct = pivot_rhna(indicator_name, df_places_sub, county, jurisdiction, columns, values, df_counties_sub, df_mpo)
    
    
        ## Plotting ---
        
        df_plot = pd.concat([df_places_sub[df_places_sub['Geography'] == jurisdiction], df_counties_sub, df_mpo])
        df_plot = df_plot.drop(values, axis=1)
        df_plot['Percentage'] = round(df_plot['Percentage'], 1)
        
        color_map = {
                 "Occupied":"#1F45FC",
                 "All vacancies": "#9DC209",
        }
        
        fig = px.bar(df_plot, x='Geography', y='Percentage'
                     , color = columns
                     , color_discrete_map=color_map)
        
        fig.update_yaxes(dtick=10, ticksuffix='%', range = [0,102])
        fig.update_layout(legend={'traceorder': 'reversed'})
        fig.update_traces(hovertemplate="%{y}")
    
        path_plots = path_out / county.replace(' County', '') / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            
            
list_indicators.append(indicator_name)



In [ ]:


indicator_name = 'RHNA_HSG_3'


# Set indicator
source = 'ACS5'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Households'
columns = 'Variable'


## Importing ---

df_places, df_counties, df_mpo = import_rhna(path_raw, indicator_name)


## Organizing ---

df_places, df_counties, df_mpo = clean_rhna(df_places, df_counties, df_mpo, path_config0, columns, values)


counties = df_counties['Geography'].unique()

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_places_sub, df_counties_sub = sub_rhna(df_places, df_counties, county)
    jurisdictions = df_places_sub['Geography'].unique()
    
    for jurisdiction in tqdm(jurisdictions):
                
        df_prod, df_pct = pivot_rhna(indicator_name, df_places_sub, county, jurisdiction, columns, values, df_counties_sub, df_mpo)
    
    
        ## Plotting ---
        
        df_plot = pd.concat([df_places_sub[df_places_sub['Geography'] == jurisdiction], df_counties_sub, df_mpo])
        df_plot = df_plot.drop(values, axis=1)
        df_plot['Percentage'] = round(df_plot['Percentage'], 1)
        
        color_map = {
                "For rent":"#1F45FC",
                "For sale only":"#9DC209",
                "For seasonal, recreational, or occasional use":"#1E90FF",
                "Other vacant":"#FBB117",
                "Rented, not occupied":"#7E587E",
                "Sold, not occupied":"#DC381F",
                "For migrant workers":"#006A4E"
        }
        
        fig = px.bar(df_plot, x='Geography', y='Percentage'
                     , color = columns
                     , color_discrete_map=color_map)
        
        fig.update_yaxes(dtick=10, ticksuffix='%', range = [0,102])
        fig.update_layout(legend={'traceorder': 'reversed'})
        fig.update_traces(hovertemplate="%{y}")
    
        path_plots = path_out / county.replace(' County', '') / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            

list_indicators.append(indicator_name)



In [ ]:


indicator_name = 'RHNA_HSG_4'


# Set indicator
source = 'ACS5'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Housing Units'


## Importing ---

df_places, df_counties, df_mpo = import_rhna(path_raw, indicator_name)



## Organizing ---

df_places['NAME'] = df_places['NAME'].str.replace(' CDP, California' , '', regex=True)
df_places['NAME'] = df_places['NAME'].str.replace(' city, California', '', regex=True)
df_places = df_places.rename(columns={'NAME':'Geography'})
df_places = df_places[df_places['Year'] == df_places['Year'].max()]
df_places = df_places.reset_index(drop=True)

df_places = df_places[['County Name', 'Geography', values, 'Variable', 'Percentage']].drop_duplicates()

df_places['Sort'] = pd.Categorical(df_places['Variable'], ['Built 2010 or later', 'Built 2000 to 2009', 'Built 1980 to 1999'
                                                           , 'Built 1960 to 1979', 'Built 1940 to 1959', 'Built 1939 or earlier'])
df_places = df_places.sort_values(['County Name', 'Geography', 'Sort'], ascending=[True, True, False])
df_places = df_places.drop(['Sort'], axis = 1)

counties = list(df_places['County Name'].unique())


for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_places_sub = df_places.copy()
    df_places_sub = df_places_sub[df_places_sub['County Name'] == county]
    jurisdictions = df_places_sub['Geography'].unique()
    
    for jurisdiction in tqdm(jurisdictions):

        df_prod = df_places_sub[df_places_sub['Geography'] == jurisdiction]
        df_prod = df_prod[['Variable', 'Housing Units', 'Percentage']].drop_duplicates()
        df_plot = df_prod.copy()

        ## Plotting ---

        fig = px.bar(df_plot, x='Variable', y=values)
        fig.update_traces(marker_color='#1E90FF')
        fig.update_traces(hovertemplate="%{y}")

        path_plots = path_out / county.replace(' County', '') / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
        
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod)
            

# list_indicators.append(indicator_name)



In [ ]:


indicator_name = 'RHNA_HSG_7'


# Set indicator
source = 'ACS5'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Households'
columns = 'Variable'


## Importing ---

df_places, df_counties, df_mpo = import_rhna(path_raw, indicator_name)


## Organizing ---

df_places, df_counties, df_mpo = clean_rhna(df_places, df_counties, df_mpo, path_config0, columns, values)


counties = df_counties['Geography'].unique()

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_places_sub, df_counties_sub = sub_rhna(df_places, df_counties, county)
    jurisdictions = df_places_sub['Geography'].unique()
    
    for jurisdiction in tqdm(jurisdictions):
                
        df_prod, df_pct = pivot_rhna(indicator_name, df_places_sub, county, jurisdiction, columns, values, df_counties_sub, df_mpo)
    
    
        ## Plotting ---
        
        df_plot = pd.concat([df_places_sub[df_places_sub['Geography'] == jurisdiction], df_counties_sub, df_mpo])
        df_plot = df_plot.drop(values, axis=1)
        df_plot['Percentage'] = round(df_plot['Percentage'], 1)
        
        color_map = {
                "Units valued less than 250k":"#1F45FC",
                "Units valued 250k-500k":"#1E90FF",
                "Units valued 500k-750k":"#9DC209",
                "Units valued 750k-1M":"#FBB117",
                "Units valued 1M-1.5M":"#7E587E",
                "Units valued 1.5M-2M":"#DC381F",
                "Units valued 2M+":"#006A4E"
        }
        
        fig = px.bar(df_plot, x='Geography', y='Percentage'
                     , color = columns
                     , color_discrete_map=color_map)
        
        fig.update_yaxes(dtick=10, ticksuffix='%', range = [0,102])
        fig.update_layout(legend={'traceorder': 'reversed'})
        fig.update_traces(hovertemplate="%{y}")
    
        path_plots = path_out / county.replace(' County', '') / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            

list_indicators.append(indicator_name)



In [ ]:


indicator_name = 'RHNA_HSG_9'


# Set indicator
source = 'ACS5'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Households'
columns = 'Variable'


## Importing ---

df_places, df_counties, df_mpo = import_rhna(path_raw, indicator_name)


## Organizing ---

df_places, df_counties, df_mpo = clean_rhna(df_places, df_counties, df_mpo, path_config0, columns, values)


counties = df_counties['Geography'].unique()

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_places_sub, df_counties_sub = sub_rhna(df_places, df_counties, county)
    jurisdictions = df_places_sub['Geography'].unique()
    
    for jurisdiction in tqdm(jurisdictions):
                
        df_prod, df_pct = pivot_rhna(indicator_name, df_places_sub, county, jurisdiction, columns, values, df_counties_sub, df_mpo)
    
    
        ## Plotting ---
        
        df_plot = pd.concat([df_places_sub[df_places_sub['Geography'] == jurisdiction], df_counties_sub, df_mpo])
        df_plot = df_plot.drop(values, axis=1)
        df_plot['Percentage'] = round(df_plot['Percentage'], 1)
        
        color_map = {
                "Rent less than 500":"#1F45FC",
                "Rent 500-1,000":"#1E90FF",
                "Rent 1,000-1,500":"#9DC209",
                "Rent 1,500-2,000":"#FBB117",
                "Rent 2,000-2,500":"#7E587E",
                "Rent 2,500-3,000":"#DC381F",
                "Rent 3,000 or more":"#006A4E"
        }
        
        fig = px.bar(df_plot, x='Geography', y='Percentage'
                     , color = columns
                     , color_discrete_map=color_map)
        
        fig.update_yaxes(dtick=10, ticksuffix='%', range = [0,102])
        fig.update_layout(legend={'traceorder': 'reversed'})
        fig.update_traces(hovertemplate="%{y}")
    
        path_plots = path_out / county.replace(' County', '') / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            

list_indicators.append(indicator_name)



***

OVER

***

In [ ]:


indicator_name = 'RHNA_OVER_1'


# Set indicator
source = 'CHAS'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Households'
columns = 'Severity'


## Importing ---

file_chas = path_raw / f'HUD_CHAS_2017thru2021.csv'
df_chas = pd.read_csv(file_chas, dtype=str)


## Organizing ---

df_chas['Households'] = df_chas['Households'].astype(int)

estimates = ['T10_est3', 'T10_est24', 'T10_est45', 'T10_est67', 'T10_est88', 'T10_est109']

df_chas = df_chas[df_chas['Estimate'].isin(estimates)]
print(df_chas['Description 1'].unique())
print(df_chas['Description 2'].unique())
print(df_chas['Description 3'].unique())
print(df_chas['Description 4'].unique())


df_chas = df_chas[['County Name', 'name', 'Description 1', 'Description 2', 'Households']]
df_chas = df_chas.rename(columns = {'Description 1':'Tenure', 'Description 2':'Severity'})
df_chas['name'] = df_chas['name'].str.replace(' city, California', '', regex=True)
df_chas['name'] = df_chas['name'].str.replace(' town, California', '', regex=True)

conditions = [
      df_chas['Severity'] == ' AND persons per room is less than or equal to 1'
    , df_chas['Severity'] == ' AND persons per room is greater than 1 but less than or equal to 1.5'
    , df_chas['Severity'] == ' AND persons per room is greater than 1.5'
]

choices = ['Less than or equal to 1 person per room', '1 to 1.5 occupants per room', 'More than 1.5 occupants per room']

df_chas['Severity'] = np.select(conditions, choices, default='no')
df_chas['Percentage'] = 100 * (df_chas['Households'] / df_chas.groupby(['County Name', 'name', 'Tenure'])['Households'].transform('sum'))
df_chas = df_chas[df_chas['Severity'] != 'Less than or equal to 1 person per room']
df_chas = df_chas.reset_index(drop=True)

display(df_chas)


counties = list(df_chas['County Name'].unique())

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_chas_sub = df_chas[df_chas['County Name'] == county]
    jurisdictions = df_chas_sub['name'].unique()
    
    for jurisdiction in tqdm(jurisdictions):

        df_prod = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        df_prod = df_prod.drop('Percentage', axis=1)
        df_prod = df_prod.pivot_table(index=['Tenure'], columns=columns, values=values).reset_index()

        df_pct = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        df_pct = df_pct.drop('Households', axis=1)
        df_pct = df_pct.pivot_table(index=['Tenure'], columns=columns, values='Percentage').reset_index()
        
        ## Plotting ---

        df_plot = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        
        color_map  = {
            '1 to 1.5 occupants per room': '#9DC209'
            , 'More than 1.5 occupants per room': '#1F45FC'
        }

        fig = px.bar(df_plot, x='Tenure', y=values
                     , color = columns
                     , barmode='group'
                     , color_discrete_map=color_map)
        
        fig.update_traces(hovertemplate="%{y}")
    
        path_plots = path_out / county / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            

list_indicators.append(indicator_name)



In [ ]:


indicator_name = 'RHNA_OVER_2'


# Set indicator
source = 'CHAS'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Households'
columns = 'Severity'


## Importing ---

file_chas = path_raw / f'HUD_CHAS_2017thru2021.csv'
df_chas = pd.read_csv(file_chas, dtype=str)


## Organizing ---

df_chas['Households'] = df_chas['Households'].astype(int)

estimates = ['T10_est3', 'T10_est24', 'T10_est45', 'T10_est67', 'T10_est88', 'T10_est109']

df_chas = df_chas[df_chas['Estimate'].isin(estimates)]
print(df_chas['Description 1'].unique())
print(df_chas['Description 2'].unique())
print(df_chas['Description 3'].unique())
print(df_chas['Description 4'].unique())


df_chas = df_chas[['County Name', 'name', 'Description 2', 'Households']]
df_chas = df_chas.rename(columns = {'Description 2':'Severity'})
df_chas['name'] = df_chas['name'].str.replace(' city, California', '', regex=True)
df_chas['name'] = df_chas['name'].str.replace(' town, California', '', regex=True)

conditions = [
      df_chas['Severity'] == ' AND persons per room is less than or equal to 1'
    , df_chas['Severity'] == ' AND persons per room is greater than 1 but less than or equal to 1.5'
    , df_chas['Severity'] == ' AND persons per room is greater than 1.5'
]

choices = ['Less than or equal to 1 person per room', 'More than 1 occupants per room', 'More than 1 occupants per room']

df_chas['Severity'] = np.select(conditions, choices, default='no')
df_chas = df_chas.reset_index(drop=True)

df_chas     = df_chas.groupby(['County Name', 'name', 'Severity'], as_index=False)['Households'].sum()
df_counties = df_chas.groupby(['County Name',         'Severity'], as_index=False)['Households'].sum()
df_mpo      = df_chas.groupby([                       'Severity'], as_index=False)['Households'].sum()
df_mpo['MPO'] = 'SACOG'

df_chas    ['Percentage'] = round(100 * (df_chas    ['Households'] / df_chas    .groupby(['County Name', 'name'])['Households'].transform('sum')), 1)
df_counties['Percentage'] = round(100 * (df_counties['Households'] / df_counties.groupby(['County Name'        ])['Households'].transform('sum')), 1)
df_mpo     ['Percentage'] = round(100 * (df_mpo     ['Households'] / df_mpo     .groupby(['MPO'                ])['Households'].transform('sum')), 1)

df_chas     = df_chas    .rename(columns = {'name':'Geography'})
df_counties = df_counties.rename(columns = {'County Name':'Geography'})
df_mpo      = df_mpo     .rename(columns = {'MPO':'Geography'})

df_chas    = df_chas    .reset_index(drop=True)
df_counties= df_counties.reset_index(drop=True)
df_mpo     = df_mpo     .reset_index(drop=True)


counties = list(df_chas['County Name'].unique())

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_counties_sub = df_counties[df_counties['Geography'  ] == county]
    df_counties_sub['Geography'] = df_counties_sub['Geography'] + ' County'
    df_chas_sub     = df_chas    [df_chas    ['County Name'] == county]

    df_chas_sub = df_chas_sub.drop(['County Name'], axis=1)
    jurisdictions = df_chas_sub['Geography'].unique()
    
    for jurisdiction in tqdm(jurisdictions):

        df_prod = pd.concat([df_chas_sub[df_chas_sub['Geography'] == jurisdiction], df_counties_sub, df_mpo])
        df_prod = df_prod.drop('Percentage', axis=1)
        df_prod = df_prod.pivot_table(index=['Geography'], columns=columns, values=values).reset_index()

        df_pct = pd.concat([df_chas_sub[df_chas_sub['Geography'] == jurisdiction], df_counties_sub, df_mpo])
        df_pct = df_pct.drop('Households', axis=1)
        df_pct = df_pct.pivot_table(index=['Geography'], columns=columns, values='Percentage').reset_index()
        
        ## Plotting ---

        df_plot = pd.concat([df_chas_sub[df_chas_sub['Geography'] == jurisdiction], df_counties_sub, df_mpo])
        
        color_map  = {
            'Less than or equal to 1 person per room': '#9DC209'
            , 'More than 1 occupants per room': '#1F45FC'
        }

        fig = px.bar(df_plot, x='Geography', y='Percentage'
                     , color = columns
                     , color_discrete_map=color_map)
        
        fig.update_traces(hovertemplate="%{y}")
    
        path_plots = path_out / county / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            

list_indicators.append(indicator_name)


In [ ]:


indicator_name = 'RHNA_OVER_4'


# Set indicator
source = 'CHAS'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Households'
columns = 'Severity'


## Importing ---

file_chas = path_raw / f'HUD_CHAS_2017thru2021.csv'
df_chas = pd.read_csv(file_chas, dtype=str)


## Organizing ---

df_chas['Households'] = df_chas['Households'].astype(int)

estimates = [
    'T10_est4', 'T10_est8', 'T10_est12', 'T10_est16', 'T10_est20', 'T10_est25', 'T10_est29', 'T10_est33',
    'T10_est37', 'T10_est41', 'T10_est46', 'T10_est50', 'T10_est54', 'T10_est58', 'T10_est62', 'T10_est68',
    'T10_est72', 'T10_est76', 'T10_est80', 'T10_est84', 'T10_est89', 'T10_est93', 'T10_est97', 'T10_est101',
    'T10_est105', 'T10_est110', 'T10_est114', 'T10_est118', 'T10_est122', 'T10_est126'
]

df_chas = df_chas[df_chas['Estimate'].isin(estimates)]
print(df_chas['Description 1'].unique())
print(df_chas['Description 2'].unique())
print(df_chas['Description 3'].unique())
print(df_chas['Description 4'].unique())


df_chas = df_chas[['County Name', 'name', 'Description 2', 'Description 3', 'Households']]
df_chas = df_chas.rename(columns = {'Description 3':'Income Level', 'Description 2':'Severity'})
df_chas['name'] = df_chas['name'].str.replace(' city, California', '', regex=True)
df_chas['name'] = df_chas['name'].str.replace(' town, California', '', regex=True)

conditions = [
      df_chas['Severity'] == ' AND persons per room is less than or equal to 1'
    , df_chas['Severity'] == ' AND persons per room is greater than 1 but less than or equal to 1.5'
    , df_chas['Severity'] == ' AND persons per room is greater than 1.5'
]

choices = ['Less than or equal to 1 person per room', '1 to 1.5 occupants per room', 'More than 1.5 occupants per room']

df_chas['Severity'] = np.select(conditions, choices, default='no')

conditions = [
    df_chas['Income Level'  ] == ' AND household income is less than or equal to 30% of HAMFI'
    , df_chas['Income Level'] == ' AND household income is greater than 30% but less than or equal to 50% of HAMFI'
    , df_chas['Income Level'] == ' AND household income is greater than 50% but less than or equal to 80% of HAMFI'
    , df_chas['Income Level'] == ' AND household income is greater than 80% but less than or equal to 100% of HAMFI'
    , df_chas['Income Level'] == ' AND household income is greater than 100% of HAMFI'
]

choices = ['0%-30% of AMI', '31%-50% of AMI', '51%-80% of AMI', '81%-100% of AMI', 'Greater than 100% of AMI']

df_chas['Income Level'] = np.select(conditions, choices, default='no')


df_chas = df_chas.groupby(['County Name', 'name', 'Income Level', 'Severity'], as_index=False)['Households'].sum()

df_chas['Percentage'] = round(100 * (df_chas['Households'] / df_chas.groupby(['County Name', 'name', 'Income Level'])['Households'].transform('sum')), 1)
df_chas = df_chas[df_chas['Severity'] != 'Less than or equal to 1 person per room']


df_chas = df_chas.reset_index(drop=True)



counties = list(df_chas['County Name'].unique())

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_chas_sub = df_chas[df_chas['County Name'] == county]
    jurisdictions = df_chas_sub['name'].unique()
    
    for jurisdiction in tqdm(jurisdictions):

        df_prod = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        df_prod = df_prod.drop('Percentage', axis=1)
        df_prod = df_prod.pivot_table(index=['Income Level'], columns=columns, values=values).reset_index()

        df_pct = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        df_pct = df_pct.drop('Households', axis=1)
        df_pct = df_pct.pivot_table(index=['Income Level'], columns=columns, values='Percentage').reset_index()
        
        ## Plotting ---

        df_plot = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        
        color_map  = {
            '1 to 1.5 occupants per room': '#9DC209'
            , 'More than 1.5 occupants per room': '#1F45FC'
        }

        fig = px.bar(df_plot, x='Income Level', y='Percentage'
                     , color = columns
                     , barmode='group'
                     , color_discrete_map=color_map)
        
        fig.update_traces(hovertemplate="%{y}")
        fig.update_yaxes(ticksuffix='%')

        path_plots = path_out / county / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            

list_indicators.append(indicator_name)




In [ ]:


indicator_name = 'RHNA_OVER_5'


# Set indicator
source = 'CHAS'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Households'
columns = 'Cost Burden'


## Importing ---

file_chas = path_raw / f'HUD_CHAS_2017thru2021.csv'
df_chas = pd.read_csv(file_chas, dtype=str)


## Organizing ---

df_chas['Households'] = df_chas['Households'].astype(int)

estimates = [
    'T8_est4', 'T8_est7', 'T8_est10', 'T8_est13', 'T8_est17', 'T8_est20', 'T8_est23', 'T8_est26', 
    'T8_est30', 'T8_est33', 'T8_est36', 'T8_est39', 'T8_est43', 'T8_est46', 'T8_est49', 'T8_est52', 
    'T8_est56', 'T8_est59', 'T8_est62', 'T8_est65', 'T8_est70', 'T8_est73', 'T8_est76', 'T8_est79',
    'T8_est83', 'T8_est86', 'T8_est89', 'T8_est92', 'T8_est96', 'T8_est99', 'T8_est102', 'T8_est105',
    'T8_est109', 'T8_est112', 'T8_est115', 'T8_est118', 'T8_est122', 'T8_est125', 'T8_est128', 'T8_est131'
]

df_chas = df_chas[df_chas['Estimate'].isin(estimates)]
print(df_chas['Description 1'].unique())
print(df_chas['Description 2'].unique())
print(df_chas['Description 3'].unique())
print(df_chas['Description 4'].unique())


df_chas = df_chas[['County Name', 'name', 'Description 2', 'Description 3', 'Households']]
df_chas = df_chas.rename(columns = {'Description 2':'Income Level', 'Description 3':'Cost Burden'})
df_chas['name'] = df_chas['name'].str.replace(' city, California', '', regex=True)
df_chas['name'] = df_chas['name'].str.replace(' town, California', '', regex=True)

conditions = [
      df_chas['Cost Burden'] == ' AND housing cost burden is less than or equal to 30%'
    , df_chas['Cost Burden'] == ' AND housing cost burden is greater than 30% but less than or equal to 50%'
    , df_chas['Cost Burden'] == ' AND housing cost burden is greater than 50%'
    , df_chas['Cost Burden'] == ' AND housing cost burden not computed (no/negative income)'
]

choices = ['0%-30% of income used for housing', '30%-50% of income used for housing', '50%+ of income used for housing', 'Not computed']

df_chas['Cost Burden'] = np.select(conditions, choices, default='no')

conditions = [
    df_chas['Income Level'  ] == ' AND household income is less than or equal to 30% of HAMFI'
    , df_chas['Income Level'] == ' AND household income is greater than 30% but less than or equal to 50% of HAMFI'
    , df_chas['Income Level'] == ' AND household income is greater than 50% but less than or equal to 80% of HAMFI'
    , df_chas['Income Level'] == ' AND household income is greater than 80% but less than or equal to 100% of HAMFI'
    , df_chas['Income Level'] == ' AND household income is greater than 100% of HAMFI'
]

choices = ['0%-30% of AMI', '31%-50% of AMI', '51%-80% of AMI', '81%-100% of AMI', 'Greater than 100% of AMI']

df_chas['Income Level'] = np.select(conditions, choices, default='no')


df_chas = df_chas.groupby(['County Name', 'name', 'Income Level', 'Cost Burden'], as_index=False)['Households'].sum()

df_chas['Percentage'] = round(100 * (df_chas['Households'] / df_chas.groupby(['County Name', 'name', 'Income Level'])['Households'].transform('sum')), 1)
df_chas = df_chas[df_chas['Cost Burden'] != 'Not computed']


df_chas = df_chas.reset_index(drop=True)


counties = list(df_chas['County Name'].unique())

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_chas_sub = df_chas[df_chas['County Name'] == county]
    jurisdictions = df_chas_sub['name'].unique()
    
    for jurisdiction in tqdm(jurisdictions):

        df_prod = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        df_prod = df_prod.drop('Percentage', axis=1)
        df_prod = df_prod.pivot_table(index=['Income Level'], columns=columns, values=values).reset_index()

        df_pct = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        df_pct = df_pct.drop('Households', axis=1)
        df_pct = df_pct.pivot_table(index=['Income Level'], columns=columns, values='Percentage').reset_index()
        
        ## Plotting ---

        df_plot = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        
        color_map  = {
            '0%-30% of income used for housing': '#1F45FC'
            , '30%-50% of income used for housing': '#1E90FF'
            , '50%+ of income used for housing': '#9DC209'
        }

        fig = px.bar(df_plot, x='Income Level', y='Percentage'
                     , color = columns
                     # , barmode='group'
                     , color_discrete_map=color_map)
        
        fig.update_traces(hovertemplate="%{y}")
        fig.update_yaxes(ticksuffix='%')
        fig.update_layout(legend={'traceorder': 'reversed'})

        path_plots = path_out / county / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            

list_indicators.append(indicator_name)



In [ ]:


indicator_name = 'RHNA_OVER_7'


# Set indicator
source = 'ACS5'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Households'
columns = 'Variable'


## Importing ---

df_places, df_counties, df_mpo = import_rhna(path_raw, indicator_name)


## Organizing ---

df_places, df_counties, df_mpo = clean_rhna(df_places, df_counties, df_mpo, path_config0, columns, values)


counties = df_counties['Geography'].unique()

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_places_sub, df_counties_sub = sub_rhna(df_places, df_counties, county)
    jurisdictions = df_places_sub['Geography'].unique()
    
    for jurisdiction in tqdm(jurisdictions):
                
        df_prod, df_pct = pivot_rhna(indicator_name, df_places_sub, county, jurisdiction, columns, values, df_counties_sub, df_mpo)
    
    
        ## Plotting ---
        
        df_plot = pd.concat([df_places_sub[df_places_sub['Geography'] == jurisdiction], df_counties_sub, df_mpo])
        df_plot = df_plot.drop(values, axis=1)
        df_plot['Percentage'] = round(df_plot['Percentage'], 1)
        
        color_map = {
                "0%-30% of income used for housing":"#1F45FC",
                "30%-50% of income used for housing":"#1E90FF",
                "50% or more of income used for housing":"#9DC209",
                "Not computed":"#FBB117"
        }
        
        fig = px.bar(df_plot, x='Geography', y='Percentage'
                     , color = columns
                     , color_discrete_map=color_map)
        
        fig.update_yaxes(dtick=10, ticksuffix='%', range = [0,102])
        fig.update_layout(legend={'traceorder': 'reversed'})
        fig.update_traces(hovertemplate="%{y}")
    
        path_plots = path_out / county.replace(' County', '') / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            
            
list_indicators.append(indicator_name)



In [ ]:


indicator_name = 'RHNA_OVER_8'


# Set indicator
source = 'CHAS'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Households'
columns = 'Cost Burden'


## Importing ---

file_chas = path_raw / f'HUD_CHAS_2017thru2021.csv'
df_chas = pd.read_csv(file_chas, dtype=str)


## Organizing ---

df_chas['Households'] = df_chas['Households'].astype(int)

estimates = [
    'T9_est4', 'T9_est5', 'T9_est6', 'T9_est7', 'T9_est9', 'T9_est10', 'T9_est11', 'T9_est12',
    'T9_est14', 'T9_est15', 'T9_est16', 'T9_est17', 'T9_est19', 'T9_est20', 'T9_est21', 'T9_est22',
    'T9_est24', 'T9_est25', 'T9_est26', 'T9_est27', 'T9_est29', 'T9_est30', 'T9_est31', 'T9_est32',
    'T9_est34', 'T9_est35', 'T9_est36', 'T9_est37', 'T9_est40', 'T9_est41', 'T9_est42', 'T9_est43',
    'T9_est45', 'T9_est46', 'T9_est47', 'T9_est48', 'T9_est50', 'T9_est51', 'T9_est52', 'T9_est53',
    'T9_est55', 'T9_est56', 'T9_est57', 'T9_est58', 'T9_est60', 'T9_est61', 'T9_est62', 'T9_est63',
    'T9_est65', 'T9_est66', 'T9_est67', 'T9_est68', 'T9_est70', 'T9_est71', 'T9_est72', 'T9_est73'
]

df_chas = df_chas[df_chas['Estimate'].isin(estimates)]
print(df_chas['Description 1'].unique())
print(df_chas['Description 2'].unique())
print(df_chas['Description 3'].unique())
print(df_chas['Description 4'].unique())


df_chas = df_chas[['County Name', 'name', 'Description 2', 'Description 3', 'Households']]
df_chas = df_chas.rename(columns = {'Description 2':'Race Ethnicity', 'Description 3':'Cost Burden'})
df_chas['name'] = df_chas['name'].str.replace(' city, California', '', regex=True)
df_chas['name'] = df_chas['name'].str.replace(' town, California', '', regex=True)

conditions = [
      df_chas['Cost Burden'] == ' AND housing cost burden is less than or equal to 30%'
    , df_chas['Cost Burden'] == ' AND housing cost burden is greater than 30% but less than or equal to 50%'
    , df_chas['Cost Burden'] == ' AND housing cost burden is greater than 50%'
    , df_chas['Cost Burden'] == ' AND housing cost burden not computed (no/negative income)'
]

choices = ['0%-30% of income used for housing', '30%-50% of income used for housing', '50%+ of income used for housing', 'Not computed']

df_chas['Cost Burden'] = np.select(conditions, choices, default='no')


conditions = [
      df_chas['Race Ethnicity'] == ' AND race/ethnicity is American Indian or Alaska Native alone, non-Hispanic'
    , df_chas['Race Ethnicity'] == ' AND race/ethnicity is Asian alone, non-Hispanic'
    , df_chas['Race Ethnicity'] == ' AND race/ethnicity is Black or African-American alone, non-Hispanic'
    , df_chas['Race Ethnicity'] == ' AND race/ethnicity is White alone, non-Hispanic'
    , df_chas['Race Ethnicity'] == ' AND race/ethnicity is Hispanic, any race'
    , df_chas['Race Ethnicity'] == ' AND race/ethnicity is Pacific Islander alone, non-Hispanic'
    , df_chas['Race Ethnicity'] == ' AND race/ethnicity is other (including multiple races, non-Hispanic)'
]

choices = ['American Indian or Alaska Native (NH)', 'Asian (NH)', 'Black or African American (NH)', 'White (NH)', 'Hispanic or Latino', 'Other race or multiple races (NH)', 'Other race or multiple races (NH)']

df_chas['Race Ethnicity'] = np.select(conditions, choices, default='no')


df_chas = df_chas.groupby(['County Name', 'name', 'Race Ethnicity', 'Cost Burden'], as_index=False)['Households'].sum()
df_chas['Percentage'] = round(100 * (df_chas['Households'] / df_chas.groupby(['County Name', 'name', 'Race Ethnicity'])['Households'].transform('sum')), 1)
df_chas = df_chas[df_chas['Cost Burden'] != 'Not computed']


df_chas = df_chas.reset_index(drop=True)


counties = list(df_chas['County Name'].unique())

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_chas_sub = df_chas[df_chas['County Name'] == county]
    jurisdictions = df_chas_sub['name'].unique()
    
    for jurisdiction in tqdm(jurisdictions):

        df_prod = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        df_prod = df_prod.drop('Percentage', axis=1)
        df_prod = df_prod.pivot_table(index=['Race Ethnicity'], columns=columns, values=values).reset_index()

        df_pct = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        df_pct = df_pct.drop('Households', axis=1)
        df_pct = df_pct.pivot_table(index=['Race Ethnicity'], columns=columns, values='Percentage').reset_index()
        
        ## Plotting ---

        df_plot = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        
        color_map  = {
            '0%-30% of income used for housing': '#1F45FC'
            , '30%-50% of income used for housing': '#1E90FF'
            , '50%+ of income used for housing': '#9DC209'
        }

        fig = px.bar(df_plot, x='Race Ethnicity', y='Percentage'
                     , color = columns
                     # , barmode='group'
                     , color_discrete_map=color_map)
        
        fig.update_traces(hovertemplate="%{y}")
        fig.update_yaxes(ticksuffix='%')
        fig.update_layout(legend={'traceorder': 'reversed'})

        path_plots = path_out / county / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            

list_indicators.append(indicator_name)



In [ ]:


indicator_name = 'RHNA_OVER_9'


# Set indicator
source = 'CHAS'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Households'
columns = 'Cost Burden'


## Importing ---

file_chas = path_raw / f'HUD_CHAS_2017thru2021.csv'
df_chas = pd.read_csv(file_chas, dtype=str)


## Organizing ---

df_chas['Households'] = df_chas['Households'].astype(int)

estimates = [
    'T7_est5', 'T7_est6', 'T7_est7', 'T7_est9', 'T7_est10', 'T7_est11', 'T7_est13', 'T7_est14', 'T7_est15', 'T7_est17',
    'T7_est18', 'T7_est19', 'T7_est21', 'T7_est22', 'T7_est23', 'T7_est26', 'T7_est27', 'T7_est28', 'T7_est30', 'T7_est31',
    'T7_est32', 'T7_est34', 'T7_est35', 'T7_est36', 'T7_est38', 'T7_est39', 'T7_est40', 'T7_est42', 'T7_est43', 'T7_est44',
    'T7_est47', 'T7_est48', 'T7_est49', 'T7_est51', 'T7_est52', 'T7_est53', 'T7_est55', 'T7_est56', 'T7_est57', 'T7_est59',
    'T7_est60', 'T7_est61', 'T7_est63', 'T7_est64', 'T7_est65', 'T7_est68', 'T7_est69', 'T7_est70', 'T7_est72', 'T7_est73',
    'T7_est74', 'T7_est76', 'T7_est77', 'T7_est78', 'T7_est80', 'T7_est81', 'T7_est82', 'T7_est84', 'T7_est85', 'T7_est86',
    'T7_est89', 'T7_est90', 'T7_est91', 'T7_est93', 'T7_est94', 'T7_est95', 'T7_est97', 'T7_est98', 'T7_est99', 'T7_est101',
    'T7_est102', 'T7_est103', 'T7_est105', 'T7_est106', 'T7_est107', 'T7_est111', 'T7_est112', 'T7_est113', 'T7_est115', 'T7_est116',
    'T7_est117', 'T7_est119', 'T7_est120', 'T7_est121', 'T7_est123', 'T7_est124', 'T7_est125', 'T7_est127', 'T7_est128', 'T7_est129',
    'T7_est132', 'T7_est133', 'T7_est134', 'T7_est136', 'T7_est137', 'T7_est138', 'T7_est140', 'T7_est141', 'T7_est142', 'T7_est144',
    'T7_est145', 'T7_est146', 'T7_est148', 'T7_est149', 'T7_est150', 'T7_est153', 'T7_est154', 'T7_est155', 'T7_est157', 'T7_est158',
    'T7_est159', 'T7_est161', 'T7_est162', 'T7_est163', 'T7_est165', 'T7_est166', 'T7_est167', 'T7_est169', 'T7_est170', 'T7_est171',
    'T7_est174', 'T7_est175', 'T7_est176', 'T7_est178', 'T7_est179', 'T7_est180', 'T7_est182', 'T7_est183', 'T7_est184', 'T7_est186',
    'T7_est187', 'T7_est188', 'T7_est190', 'T7_est191', 'T7_est192', 'T7_est195', 'T7_est196', 'T7_est197', 'T7_est199', 'T7_est200',
    'T7_est201', 'T7_est203', 'T7_est204', 'T7_est205', 'T7_est207', 'T7_est208', 'T7_est209', 'T7_est211', 'T7_est212', 'T7_est213'
]

df_chas = df_chas[df_chas['Estimate'].isin(estimates)]
print(df_chas['Description 1'].unique())
print(df_chas['Description 2'].unique())
print(df_chas['Description 3'].unique())
print(df_chas['Description 4'].unique())


df_chas = df_chas[['County Name', 'name', 'Description 3', 'Description 4', 'Households']]
df_chas = df_chas.rename(columns = {'Description 3':'Household Size', 'Description 4':'Cost Burden'})
df_chas['name'] = df_chas['name'].str.replace(' city, California', '', regex=True)
df_chas['name'] = df_chas['name'].str.replace(' town, California', '', regex=True)

conditions = [
      df_chas['Cost Burden'] == 'housing cost burden is less than or equal to 30%'
    , df_chas['Cost Burden'] == 'housing cost burden is greater than 30% but less than or equal to 50%'
    , df_chas['Cost Burden'] == 'housing cost burden is greater than 50%'
    , df_chas['Cost Burden'] == 'housing cost burden not computed (no/negative income)'
]

choices = ['0%-30% of income used for housing', '30%-50% of income used for housing', '50%+ of income used for housing', 'Not computed']

df_chas['Cost Burden'] = np.select(conditions, choices, default='no')


conditions = [
      df_chas['Household Size'] == 'household type is elderly family (2 persons, with either or both age 62 or over)'
    , df_chas['Household Size'] == 'household type is small family (2 persons, neither person 62 years or over, or 3 or 4 persons)'
    , df_chas['Household Size'] == 'household type is large family (5 or more persons)'
    , df_chas['Household Size'] == 'household type is elderly non-family'
    , df_chas['Household Size'] == 'other household type (non-elderly non-family)'
]

choices = ['All other household types', 'All other household types', 'Large family with 5+ persons', 'All other household types', 'All other household types']

df_chas['Household Size'] = np.select(conditions, choices, default='no')


df_chas = df_chas.groupby(['County Name', 'name', 'Household Size', 'Cost Burden'], as_index=False)['Households'].sum()

df_chas['Percentage'] = round(100 * (df_chas['Households'] / df_chas.groupby(['County Name', 'name', 'Household Size'])['Households'].transform('sum')), 1)
df_chas = df_chas[df_chas['Cost Burden'] != 'Not computed']


df_chas = df_chas.reset_index(drop=True)


counties = list(df_chas['County Name'].unique())

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_chas_sub = df_chas[df_chas['County Name'] == county]
    jurisdictions = df_chas_sub['name'].unique()
    
    for jurisdiction in tqdm(jurisdictions):

        df_prod = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        df_prod = df_prod.drop('Percentage', axis=1)
        df_prod = df_prod.pivot_table(index=['Household Size'], columns=columns, values=values).reset_index()

        df_pct = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        df_pct = df_pct.drop('Households', axis=1)
        df_pct = df_pct.pivot_table(index=['Household Size'], columns=columns, values='Percentage').reset_index()
        
        ## Plotting ---

        df_plot = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        
        color_map  = {
            '0%-30% of income used for housing': '#1F45FC'
            , '30%-50% of income used for housing': '#1E90FF'
            , '50%+ of income used for housing': '#9DC209'
        }

        fig = px.bar(df_plot, x='Household Size', y='Percentage'
                     , color = columns
                     # , barmode='group'
                     , color_discrete_map=color_map)
        
        fig.update_traces(hovertemplate="%{y}")
        fig.update_yaxes(ticksuffix='%')
        fig.update_layout(legend={'traceorder': 'reversed'})

        path_plots = path_out / county / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            

list_indicators.append(indicator_name)



***

LGFEM

***

In [ ]:


indicator_name = 'RHNA_LGFEM_2'


# Set indicator
source = 'ACS5'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Households'
columns = 'Variable'


## Importing ---

df_places, df_counties, df_mpo = import_rhna(path_raw, indicator_name)


## Organizing ---

df_places, df_counties, df_mpo = clean_rhna(df_places, df_counties, df_mpo, path_config0, columns, values)


counties = df_counties['Geography'].unique()

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_places_sub, df_counties_sub = sub_rhna(df_places, df_counties, county)
    jurisdictions = df_places_sub['Geography'].unique()
    
    for jurisdiction in tqdm(jurisdictions):
                
        df_prod, df_pct = pivot_rhna(indicator_name, df_places_sub, county, jurisdiction, columns, values, df_counties_sub, df_mpo)
    
    
        ## Plotting ---
        
        df_plot = pd.concat([df_places_sub[df_places_sub['Geography'] == jurisdiction], df_counties_sub, df_mpo])
        df_plot = df_plot.drop(values, axis=1)
        df_plot['Percentage'] = round(df_plot['Percentage'], 1)
        
        color_map = {
                "1 person households":"#1F45FC",
                "2 person households":"#1E90FF",
                "3-4 person households":"#9DC209",
                "5 or more person households":"#FBB117"
        }
        
        fig = px.bar(df_plot, x='Geography', y='Percentage'
                     , color = columns
                     , color_discrete_map=color_map)
        
        fig.update_yaxes(dtick=10, ticksuffix='%', range = [0,102])
        fig.update_layout(legend={'traceorder': 'reversed'})
        fig.update_traces(hovertemplate="%{y}")
    
        path_plots = path_out / county.replace(' County', '') / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            
            
list_indicators.append(indicator_name)



In [ ]:


indicator_name = 'RHNA_LGFEM_3'


# Set indicator
source = 'CHAS'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Households'
columns = 'Income Level'


## Importing ---

file_chas = path_raw / f'HUD_CHAS_2017thru2021.csv'
df_chas = pd.read_csv(file_chas, dtype=str)


## Organizing ---

df_chas['Households'] = df_chas['Households'].astype(int)

estimates = [
    'T7_est5', 'T7_est6', 'T7_est7', 'T7_est9', 'T7_est10', 'T7_est11', 'T7_est13', 'T7_est14', 'T7_est15', 'T7_est17',
    'T7_est18', 'T7_est19', 'T7_est21', 'T7_est22', 'T7_est23', 'T7_est26', 'T7_est27', 'T7_est28', 'T7_est30', 'T7_est31',
    'T7_est32', 'T7_est34', 'T7_est35', 'T7_est36', 'T7_est38', 'T7_est39', 'T7_est40', 'T7_est42', 'T7_est43', 'T7_est44',
    'T7_est47', 'T7_est48', 'T7_est49', 'T7_est51', 'T7_est52', 'T7_est53', 'T7_est55', 'T7_est56', 'T7_est57', 'T7_est59',
    'T7_est60', 'T7_est61', 'T7_est63', 'T7_est64', 'T7_est65', 'T7_est68', 'T7_est69', 'T7_est70', 'T7_est72', 'T7_est73',
    'T7_est74', 'T7_est76', 'T7_est77', 'T7_est78', 'T7_est80', 'T7_est81', 'T7_est82', 'T7_est84', 'T7_est85', 'T7_est86',
    'T7_est89', 'T7_est90', 'T7_est91', 'T7_est93', 'T7_est94', 'T7_est95', 'T7_est97', 'T7_est98', 'T7_est99', 'T7_est101',
    'T7_est102', 'T7_est103', 'T7_est105', 'T7_est106', 'T7_est107', 'T7_est111', 'T7_est112', 'T7_est113', 'T7_est115', 'T7_est116',
    'T7_est117', 'T7_est119', 'T7_est120', 'T7_est121', 'T7_est123', 'T7_est124', 'T7_est125', 'T7_est127', 'T7_est128', 'T7_est129',
    'T7_est132', 'T7_est133', 'T7_est134', 'T7_est136', 'T7_est137', 'T7_est138', 'T7_est140', 'T7_est141', 'T7_est142', 'T7_est144',
    'T7_est145', 'T7_est146', 'T7_est148', 'T7_est149', 'T7_est150', 'T7_est153', 'T7_est154', 'T7_est155', 'T7_est157', 'T7_est158',
    'T7_est159', 'T7_est161', 'T7_est162', 'T7_est163', 'T7_est165', 'T7_est166', 'T7_est167', 'T7_est169', 'T7_est170', 'T7_est171',
    'T7_est174', 'T7_est175', 'T7_est176', 'T7_est178', 'T7_est179', 'T7_est180', 'T7_est182', 'T7_est183', 'T7_est184', 'T7_est186',
    'T7_est187', 'T7_est188', 'T7_est190', 'T7_est191', 'T7_est192', 'T7_est195', 'T7_est196', 'T7_est197', 'T7_est199', 'T7_est200',
    'T7_est201', 'T7_est203', 'T7_est204', 'T7_est205', 'T7_est207', 'T7_est208', 'T7_est209', 'T7_est211', 'T7_est212', 'T7_est213'
]

df_chas = df_chas[df_chas['Estimate'].isin(estimates)]
print(df_chas['Description 1'].unique())
print(df_chas['Description 2'].unique())
print(df_chas['Description 3'].unique())
print(df_chas['Description 4'].unique())


df_chas = df_chas[['County Name', 'name', 'Description 2', 'Description 3', 'Households']]
df_chas = df_chas.rename(columns = {'Description 2':'Income Level', 'Description 3':'Household Size'})
df_chas['name'] = df_chas['name'].str.replace(' city, California', '', regex=True)
df_chas['name'] = df_chas['name'].str.replace(' town, California', '', regex=True)

conditions = [
    df_chas['Income Level'  ] == 'household income is less than or equal to 30% of HAMFI'
    , df_chas['Income Level'] == 'household income is greater than 30% but less than or equal to 50% of HAMFI'
    , df_chas['Income Level'] == 'household income is greater than 50% but less than or equal to 80% of HAMFI'
    , df_chas['Income Level'] == 'household income is greater than 80% but less than or equal to 100% of HAMFI'
    , df_chas['Income Level'] == 'household income is greater than 100% of HAMFI'
]

choices = ['0%-30% of AMI', '31%-50% of AMI', '51%-80% of AMI', '81%-100% of AMI', 'Greater than 100% of AMI']

df_chas['Income Level'] = np.select(conditions, choices, default='no')


conditions = [
      df_chas['Household Size'] == 'household type is elderly family (2 persons, with either or both age 62 or over)'
    , df_chas['Household Size'] == 'household type is small family (2 persons, neither person 62 years or over, or 3 or 4 persons)'
    , df_chas['Household Size'] == 'household type is large family (5 or more persons)'
    , df_chas['Household Size'] == 'household type is elderly non-family'
    , df_chas['Household Size'] == 'other household type (non-elderly non-family)'
]

choices = ['All other household types', 'All other household types', 'Large family with 5+ persons', 'All other household types', 'All other household types']

df_chas['Household Size'] = np.select(conditions, choices, default='no')


df_chas = df_chas.groupby(['County Name', 'name', 'Household Size', 'Income Level'], as_index=False)['Households'].sum()

df_chas['Percentage'] = round(100 * (df_chas['Households'] / df_chas.groupby(['County Name', 'name', 'Household Size'])['Households'].transform('sum')), 1)


df_chas = df_chas.reset_index(drop=True)

display(df_chas)


counties = list(df_chas['County Name'].unique())

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_chas_sub = df_chas[df_chas['County Name'] == county]
    jurisdictions = df_chas_sub['name'].unique()
    
    for jurisdiction in tqdm(jurisdictions):

        df_prod = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        df_prod = df_prod.drop('Percentage', axis=1)
        df_prod = df_prod.pivot_table(index=['Household Size'], columns=columns, values=values).reset_index()

        df_pct = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        df_pct = df_pct.drop('Households', axis=1)
        df_pct = df_pct.pivot_table(index=['Household Size'], columns=columns, values='Percentage').reset_index()
        
        ## Plotting ---

        df_plot = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        
        color_map  = {
            '0%-30% of AMI': '#1F45FC'
            , '31%-50% of AMI': '#1E90FF'
            , '51%-80% of AMI': '#9DC209'
            , '81%-100% of AMI': '#7E587E'
            , 'Greater than 100% of AMI': '#FBB117'
        }

        fig = px.bar(df_plot, x='Household Size', y='Percentage'
                     , color = columns
                     # , barmode='group'
                     , color_discrete_map=color_map)
        
        fig.update_traces(hovertemplate="%{y}")
        fig.update_yaxes(ticksuffix='%')
        fig.update_layout(legend={'traceorder': 'reversed'})

        path_plots = path_out / county / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            

list_indicators.append(indicator_name)



***

DISAB

***

In [ ]:


indicator_name = 'RHNA_DISAB_2'


# Set indicator
source = 'ACS5'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Population'
columns = 'Variable'


## Importing ---

df_places, df_counties, df_mpo = import_rhna(path_raw, indicator_name)


## Organizing ---

df_places, df_counties, df_mpo = clean_rhna(df_places, df_counties, df_mpo, path_config0, columns, values)


counties = df_counties['Geography'].unique()

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_places_sub, df_counties_sub = sub_rhna(df_places, df_counties, county)
    jurisdictions = df_places_sub['Geography'].unique()
    
    for jurisdiction in tqdm(jurisdictions):
                
        df_prod, df_pct = pivot_rhna(indicator_name, df_places_sub, county, jurisdiction, columns, values, df_counties_sub, df_mpo)
    
    
        ## Plotting ---
        
        df_plot = pd.concat([df_places_sub[df_places_sub['Geography'] == jurisdiction], df_counties_sub, df_mpo])
        df_plot = df_plot.drop(values, axis=1)
        df_plot['Percentage'] = round(df_plot['Percentage'], 1)
            
        color_map  = {
            'With a disability': '#9DC209'
            , 'No disability': '#1F45FC'
            # , 'Married-couple family households': '#1F45FC'
        }
    
        fig = px.bar(df_plot, x='Geography', y='Percentage'
                     , color = columns
                     , color_discrete_map=color_map)
        
        fig.update_yaxes(dtick=10, ticksuffix='%', range = [0,102])
        fig.update_layout(legend={'traceorder': 'reversed'})
        fig.update_traces(hovertemplate="%{y}")
    
        path_plots = path_out / county.replace(' County', '') / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            

list_indicators.append(indicator_name)



***

AFFH

***

In [ ]:


indicator_name = 'RHNA_AFFH_3'


# Set indicator
source = 'ACS5'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Population'
columns = 'Variable'


## Importing ---

df_places, df_counties, df_mpo = import_rhna(path_raw, indicator_name)


## Organizing ---


df_places, df_counties, df_mpo = clean_rhna(df_places, df_counties, df_mpo, path_config0, columns, values)


counties = df_counties['Geography'].unique()

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_places_sub, df_counties_sub = sub_rhna(df_places, df_counties, county)
    jurisdictions = df_places_sub['Geography'].unique()
    
    for jurisdiction in tqdm(jurisdictions):
                
        df_prod, df_pct = pivot_rhna(indicator_name, df_places_sub, county, jurisdiction, columns, values, df_counties_sub, df_mpo)
    
    
        ## Plotting ---
        
        df_plot = pd.concat([df_places_sub[df_places_sub['Geography'] == jurisdiction], df_counties_sub, df_mpo])
        df_plot = df_plot.drop(values, axis=1)
        df_plot['Percentage'] = round(df_plot['Percentage'], 1)
            
        color_map  = {
            'Population 5 years and over who speak english "well" or "very well"': '#1F45FC'
            , 'Population 5 years and over who speak english "not well" or "not at all"': '#9DC209'
            # , 'Married-couple family households': '#1F45FC'
        }
    
        fig = px.bar(df_plot, x='Geography', y='Percentage'
                     , color = columns
                     , color_discrete_map=color_map)
        
        fig.update_yaxes(dtick=10, ticksuffix='%', range = [0,102])
        fig.update_layout(legend={'traceorder': 'reversed'})
        fig.update_traces(hovertemplate="%{y}")
    
        path_plots = path_out / county.replace(' County', '') / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            

list_indicators.append(indicator_name)



***

SEN_1

***

In [ ]:


indicator_name = 'RHNA_SEN_1'


# Set indicator
source = 'CHAS'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Households'
columns = 'Tenure'


## Importing ---

file_chas = path_raw / f'HUD_CHAS_2017thru2021.csv'
df_chas = pd.read_csv(file_chas, dtype=str)


## Organizing ---

df_chas['Households'] = df_chas['Households'].astype(int)

estimates = [
    'T7_est16', 'T7_est37', 'T7_est58', 'T7_est79', 'T7_est100',
    'T7_est122', 'T7_est143', 'T7_est164', 'T7_est185', 'T7_est206'
]

df_chas = df_chas[df_chas['Estimate'].isin(estimates)]
print(df_chas['Description 1'].unique())
print(df_chas['Description 2'].unique())
print(df_chas['Description 3'].unique())
print(df_chas['Description 4'].unique())


df_chas = df_chas[['County Name', 'name', 'Description 1', 'Description 2', 'Households']]
df_chas = df_chas.rename(columns = {'Description 1':'Tenure', 'Description 2':'Income Level'})
df_chas['name'] = df_chas['name'].str.replace(' city, California', '', regex=True)
df_chas['name'] = df_chas['name'].str.replace(' town, California', '', regex=True)

conditions = [
    df_chas['Income Level'] == 'household income is less than or equal to 30% of HAMFI'
    , df_chas['Income Level'] == 'household income is greater than 30% but less than or equal to 50% of HAMFI'
    , df_chas['Income Level'] == 'household income is greater than 50% but less than or equal to 80% of HAMFI'
    , df_chas['Income Level'] == 'household income is greater than 80% but less than or equal to 100% of HAMFI'
    , df_chas['Income Level'] == 'household income is greater than 100% of HAMFI'
]

choices = ['0%-30% of AMI', '31%-50% of AMI', '51%-80% of AMI', '81%-100% of AMI', 'Greater than 100% of AMI']

df_chas['Income Level'] = np.select(conditions, choices, default='no')
df_chas['Percentage'] = round(100 * (df_chas['Households'] / df_chas.groupby(['County Name', 'name', 'Income Level'])['Households'].transform('sum')), 1)
df_chas = df_chas.reset_index(drop=True)



counties = list(df_chas['County Name'].unique())

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_chas_sub = df_chas[df_chas['County Name'] == county]
    jurisdictions = df_chas_sub['name'].unique()
    
    for jurisdiction in tqdm(jurisdictions):

        print()
        df_prod = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        df_prod = df_prod.drop('Percentage', axis=1)
        df_prod = df_prod.pivot_table(index=['Income Level'], columns=columns, values=values).reset_index()

        df_pct = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        df_pct = df_pct.drop('Households', axis=1)
        df_pct = df_pct.pivot_table(index=['Income Level'], columns=columns, values='Percentage').reset_index()
        
        ## Plotting ---

        df_plot = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        
        color_map = {
                 "Owner occupied":"#1F45FC",
                 "Renter occupied": "#9DC209",
        }

        fig = px.bar(df_plot, x='Income Level', y='Percentage'
                     , color = columns
                     # , barmode='group'
                     , color_discrete_map=color_map)
        
        fig.update_traces(hovertemplate="%{y}")
        fig.update_yaxes(ticksuffix='%')
        # fig.update_layout(legend={'traceorder': 'reversed'})

        path_plots = path_out / county / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            

list_indicators.append(indicator_name)



In [ ]:


indicator_name = 'RHNA_SEN_3'


# Set indicator
source = 'CHAS'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Households'
columns = 'Cost Burden'


## Importing ---

file_chas = path_raw / f'HUD_CHAS_2017thru2021.csv'
df_chas = pd.read_csv(file_chas, dtype=str)


## Organizing ---

df_chas['Households'] = df_chas['Households'].astype(int)

estimates = [
    'T7_est17', 'T7_est18', 'T7_est19', 'T7_est38', 'T7_est39', 'T7_est40', 'T7_est59', 'T7_est60', 'T7_est61',
    'T7_est80', 'T7_est81', 'T7_est82', 'T7_est101', 'T7_est102', 'T7_est103', 'T7_est123', 'T7_est124', 'T7_est125',
    'T7_est144', 'T7_est145', 'T7_est146', 'T7_est165', 'T7_est166', 'T7_est167', 'T7_est186', 'T7_est187', 'T7_est188',
    'T7_est207', 'T7_est208', 'T7_est209'
]


df_chas = df_chas[df_chas['Estimate'].isin(estimates)]
print(df_chas['Description 1'].unique())
print(df_chas['Description 2'].unique())
print(df_chas['Description 3'].unique())
print(df_chas['Description 4'].unique())


df_chas = df_chas[['County Name', 'name', 'Description 2', 'Description 4', 'Households']]
df_chas = df_chas.rename(columns = {'Description 4':'Cost Burden', 'Description 2':'Income Level'})
df_chas['name'] = df_chas['name'].str.replace(' city, California', '', regex=True)
df_chas['name'] = df_chas['name'].str.replace(' town, California', '', regex=True)

conditions = [
    df_chas['Income Level'] == 'household income is less than or equal to 30% of HAMFI'
    , df_chas['Income Level'] == 'household income is greater than 30% but less than or equal to 50% of HAMFI'
    , df_chas['Income Level'] == 'household income is greater than 50% but less than or equal to 80% of HAMFI'
    , df_chas['Income Level'] == 'household income is greater than 80% but less than or equal to 100% of HAMFI'
    , df_chas['Income Level'] == 'household income is greater than 100% of HAMFI'
]

choices = ['0%-30% of AMI', '31%-50% of AMI', '51%-80% of AMI', '81%-100% of AMI', 'Greater than 100% of AMI']

df_chas['Income Level'] = np.select(conditions, choices, default='no')


conditions = [
      df_chas['Cost Burden'] == 'housing cost burden is less than or equal to 30%'
    , df_chas['Cost Burden'] == 'housing cost burden is greater than 30% but less than or equal to 50%'
    , df_chas['Cost Burden'] == 'housing cost burden is greater than 50%'
    , df_chas['Cost Burden'] == 'housing cost burden not computed (no/negative income)'
]

choices = ['0%-30% of income used for housing', '30%-50% of income used for housing', '50%+ of income used for housing', 'Not computed']

df_chas['Cost Burden'] = np.select(conditions, choices, default='no')

df_chas = df_chas.groupby(['County Name', 'name', 'Income Level', 'Cost Burden'], as_index=False)['Households'].sum()

df_chas['Percentage'] = round(100 * (df_chas['Households'] / df_chas.groupby(['County Name', 'name', 'Income Level'])['Households'].transform('sum')), 1)
df_chas = df_chas.reset_index(drop=True)

display(df_chas)

counties = list(df_chas['County Name'].unique())

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_chas_sub = df_chas[df_chas['County Name'] == county]
    jurisdictions = df_chas_sub['name'].unique()
    
    for jurisdiction in tqdm(jurisdictions):

        df_prod = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        df_prod = df_prod.drop('Percentage', axis=1)
        df_prod = df_prod.pivot_table(index=['Income Level'], columns=columns, values=values).reset_index()

        df_pct = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        df_pct = df_pct.drop('Households', axis=1)
        df_pct = df_pct.pivot_table(index=['Income Level'], columns=columns, values='Percentage').reset_index()
        
        ## Plotting ---

        df_plot = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        
        color_map  = {
            '0%-30% of income used for housing': '#1F45FC'
            , '30%-50% of income used for housing': '#1E90FF'
            , '50%+ of income used for housing': '#9DC209'
        }

        fig = px.bar(df_plot, x='Income Level', y='Percentage'
                     , color = columns
                     # , barmode='group'
                     , color_discrete_map=color_map)
        
        fig.update_traces(hovertemplate="%{y}")
        fig.update_yaxes(ticksuffix='%')
        fig.update_layout(legend={'traceorder': 'reversed'})

        path_plots = path_out / county / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            

list_indicators.append(indicator_name)



***

ELI

***

In [ ]:


indicator_name = 'RHNA_ELI_1'


# Set indicator
source = 'CHAS'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Households'
columns = 'Income Level'


## Importing ---

file_chas = path_raw / f'HUD_CHAS_2017thru2021.csv'
df_chas = pd.read_csv(file_chas, dtype=str)


## Organizing ---

df_chas['Households'] = df_chas['Households'].astype(int)

estimates = ['T7_est3', 'T7_est24', 'T7_est45', 'T7_est66', 'T7_est87', 'T7_est109', 'T7_est130', 'T7_est151', 'T7_est172', 'T7_est193']


df_chas = df_chas[df_chas['Estimate'].isin(estimates)]
print(df_chas['Description 1'].unique())
print(df_chas['Description 2'].unique())
print(df_chas['Description 3'].unique())
print(df_chas['Description 4'].unique())


df_chas = df_chas[['County Name', 'name', 'Description 2', 'Households']]
df_chas = df_chas.rename(columns = {'Description 2':'Income Level'})
df_chas['name'] = df_chas['name'].str.replace(' city, California', '', regex=True)
df_chas['name'] = df_chas['name'].str.replace(' town, California', '', regex=True)

conditions = [
    df_chas['Income Level'] == 'household income is less than or equal to 30% of HAMFI'
    , df_chas['Income Level'] == 'household income is greater than 30% but less than or equal to 50% of HAMFI'
    , df_chas['Income Level'] == 'household income is greater than 50% but less than or equal to 80% of HAMFI'
    , df_chas['Income Level'] == 'household income is greater than 80% but less than or equal to 100% of HAMFI'
    , df_chas['Income Level'] == 'household income is greater than 100% of HAMFI'
]

choices = ['0%-30% of AMI', '31%-50% of AMI', '51%-80% of AMI', '81%-100% of AMI', 'Greater than 100% of AMI']

df_chas['Income Level'] = np.select(conditions, choices, default='no')


df_chas     = df_chas.groupby(['County Name', 'name', 'Income Level'], as_index=False)['Households'].sum()
df_counties = df_chas.groupby(['County Name',         'Income Level'], as_index=False)['Households'].sum()
df_mpo      = df_chas.groupby([                       'Income Level'], as_index=False)['Households'].sum()
df_mpo['MPO'] = 'SACOG'

df_chas    ['Percentage'] = round(100 * (df_chas    ['Households'] / df_chas    .groupby(['County Name', 'name'])['Households'].transform('sum')), 1)
df_counties['Percentage'] = round(100 * (df_counties['Households'] / df_counties.groupby(['County Name'        ])['Households'].transform('sum')), 1)
df_mpo     ['Percentage'] = round(100 * (df_mpo     ['Households'] / df_mpo     .groupby(['MPO'                ])['Households'].transform('sum')), 1)

df_chas     = df_chas    .rename(columns = {'name':'Geography'})
df_counties = df_counties.rename(columns = {'County Name':'Geography'})
df_mpo      = df_mpo     .rename(columns = {'MPO':'Geography'})

df_chas    = df_chas    .reset_index(drop=True)
df_counties= df_counties.reset_index(drop=True)
df_mpo     = df_mpo     .reset_index(drop=True)


counties = list(df_chas['County Name'].unique())

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_counties_sub              = df_counties    [df_counties['Geography'  ] == county]
    df_chas_sub                  = df_chas        [df_chas    ['County Name'] == county]
    df_counties_sub['Geography'] = df_counties_sub['Geography'] + ' County'

    df_chas_sub = df_chas_sub.drop(['County Name'], axis=1)
    jurisdictions = df_chas_sub['Geography'].unique()
    
    for jurisdiction in tqdm(jurisdictions):

        df_prod = pd.concat([df_chas_sub[df_chas_sub['Geography'] == jurisdiction], df_counties_sub, df_mpo])
        df_prod = df_prod.drop('Percentage', axis=1)
        df_prod = df_prod.pivot_table(index=['Geography'], columns=columns, values=values).reset_index()

        df_pct = pd.concat([df_chas_sub[df_chas_sub['Geography'] == jurisdiction], df_counties_sub, df_mpo])
        df_pct = df_pct.drop('Households', axis=1)
        df_pct = df_pct.pivot_table(index=['Geography'], columns=columns, values='Percentage').reset_index()
        
        ## Plotting ---

        df_plot = pd.concat([df_chas_sub[df_chas_sub['Geography'] == jurisdiction], df_counties_sub, df_mpo])
        
        color_map  = {
            '0%-30% of AMI': '#1F45FC'
            , '31%-50% of AMI': '#1E90FF'
            , '51%-80% of AMI': '#9DC209'
            , '81%-100% of AMI': '#7E587E'
            , 'Greater than 100% of AMI': '#FBB117'
        }

        fig = px.bar(df_plot, x='Geography', y='Percentage'
                     , color = columns
                     , color_discrete_map=color_map)
        
        fig.update_traces(hovertemplate="%{y}")
    
        path_plots = path_out / county / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            

list_indicators.append(indicator_name)


In [ ]:


indicator_name = 'RHNA_ELI_2'


# Set indicator
source = 'CHAS'
with path_func.open("r") as f: exec(f.read())
title = dict_about[source][indicator_name.replace('RHNA_', '')]['Indicator Title'][0]
values = 'Households'
columns = 'Income Level'


## Importing ---

file_chas = path_raw / f'HUD_CHAS_2017thru2021.csv'
df_chas = pd.read_csv(file_chas, dtype=str)


## Organizing ---

df_chas['Households'] = df_chas['Households'].astype(int)

estimates = [
    'T1_est5', 'T1_est6', 'T1_est7', 'T1_est8', 'T1_est9', 'T1_est10', 'T1_est12', 'T1_est13', 'T1_est14',
    'T1_est15', 'T1_est16', 'T1_est17', 'T1_est19', 'T1_est20', 'T1_est21', 'T1_est22', 'T1_est23', 'T1_est24',
    'T1_est26', 'T1_est27', 'T1_est28', 'T1_est29', 'T1_est30', 'T1_est31', 'T1_est33', 'T1_est34', 'T1_est35',
    'T1_est36', 'T1_est37', 'T1_est38', 'T1_est41', 'T1_est42', 'T1_est43', 'T1_est44', 'T1_est45', 'T1_est46',
    'T1_est48', 'T1_est49', 'T1_est50', 'T1_est51', 'T1_est52', 'T1_est53', 'T1_est55', 'T1_est56', 'T1_est57',
    'T1_est58', 'T1_est59', 'T1_est60', 'T1_est62', 'T1_est63', 'T1_est64', 'T1_est65', 'T1_est66', 'T1_est67',
    'T1_est69', 'T1_est70', 'T1_est71', 'T1_est72', 'T1_est73', 'T1_est74', 'T1_est78', 'T1_est79', 'T1_est80',
    'T1_est81', 'T1_est82', 'T1_est83', 'T1_est85', 'T1_est86', 'T1_est87', 'T1_est88', 'T1_est89', 'T1_est90',
    'T1_est92', 'T1_est93', 'T1_est94', 'T1_est95', 'T1_est96', 'T1_est97', 'T1_est99', 'T1_est100', 'T1_est101',
    'T1_est102', 'T1_est103', 'T1_est104', 'T1_est106', 'T1_est107', 'T1_est108', 'T1_est109', 'T1_est110', 'T1_est111',
    'T1_est114', 'T1_est115', 'T1_est116', 'T1_est117', 'T1_est118', 'T1_est119', 'T1_est121', 'T1_est122', 'T1_est123',
    'T1_est124', 'T1_est125', 'T1_est126', 'T1_est128', 'T1_est129', 'T1_est130', 'T1_est131', 'T1_est132', 'T1_est133',
    'T1_est135', 'T1_est136', 'T1_est137', 'T1_est138', 'T1_est139', 'T1_est140', 'T1_est142', 'T1_est143', 'T1_est144',
    'T1_est145', 'T1_est146', 'T1_est147'
]

df_chas = df_chas[df_chas['Estimate'].isin(estimates)]
print(df_chas['Description 1'].unique())
print(df_chas['Description 2'].unique())
print(df_chas['Description 3'].unique())
print(df_chas['Description 4'].unique())


df_chas = df_chas[['County Name', 'name', 'Description 4', 'Description 3', 'Households']]
df_chas = df_chas.rename(columns = {'Description 4':'Race Ethnicity', 'Description 3':'Income Level'})
df_chas['name'] = df_chas['name'].str.replace(' city, California', '', regex=True)
df_chas['name'] = df_chas['name'].str.replace(' town, California', '', regex=True)

conditions = [
    df_chas['Income Level'  ] == 'less than or equal to 30% of HAMFI'
    , df_chas['Income Level'] == 'greater than 30% but less than or equal to 50% of HAMFI'
    , df_chas['Income Level'] == 'greater than 50% but less than or equal to 80% of HAMFI'
    , df_chas['Income Level'] == 'greater than 80% but less than or equal to 100% of HAMFI'
    , df_chas['Income Level'] == 'greater than 100% of HAMFI'
]

choices = ['0%-30% of AMI', '31%-50% of AMI', '51%-80% of AMI', '81%-100% of AMI', 'Greater than 100% of AMI']

df_chas['Income Level'] = np.select(conditions, choices, default='no')


conditions = [
      df_chas['Race Ethnicity'] == 'American Indian or Alaska Native alone, non-Hispanic'
    , df_chas['Race Ethnicity'] == 'Asian alone, non-Hispanic'
    , df_chas['Race Ethnicity'] == 'Black or African-American alone, non-Hispanic'
    , df_chas['Race Ethnicity'] == 'White alone, non-Hispanic'
    , df_chas['Race Ethnicity'] == 'Hispanic, any race'
    , df_chas['Race Ethnicity'] == 'Pacific Islander alone, non-Hispanic'
    , df_chas['Race Ethnicity'] == 'other (including multiple races, non-Hispanic)'
]

choices = ['American Indian or Alaska Native (NH)', 'Asian (NH)', 'Black or African American (NH)', 'White (NH)', 'Hispanic or Latino', 'Other race or multiple races (NH)', 'Other race or multiple races (NH)']

df_chas['Race Ethnicity'] = np.select(conditions, choices, default='no')


df_chas = df_chas.groupby(['County Name', 'name', 'Race Ethnicity', 'Income Level'], as_index=False)['Households'].sum()
df_chas['Percentage'] = round(100 * (df_chas['Households'] / df_chas.groupby(['County Name', 'name', 'Race Ethnicity'])['Households'].transform('sum')), 1)


df_chas = df_chas.reset_index(drop=True)


counties = list(df_chas['County Name'].unique())

for county in counties:
    
    print();print()
    print(county)
    time.sleep(2)

    df_chas_sub = df_chas[df_chas['County Name'] == county]
    jurisdictions = df_chas_sub['name'].unique()
    
    for jurisdiction in tqdm(jurisdictions):

        tqdm.write(jurisdiction)

        df_prod = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        df_prod = df_prod.drop('Percentage', axis=1)
        df_prod = df_prod.pivot_table(index=['Race Ethnicity'], columns=columns, values=values).reset_index()

        df_pct = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        df_pct = df_pct.drop('Households', axis=1)
        df_pct = df_pct.pivot_table(index=['Race Ethnicity'], columns=columns, values='Percentage').reset_index()
        
        ## Plotting ---

        df_plot = df_chas_sub[df_chas_sub['name'] == jurisdiction]
        
        color_map  = {
            '0%-30% of AMI': '#1F45FC'
            , '31%-50% of AMI': '#1E90FF'
            , '51%-80% of AMI': '#9DC209'
            , '81%-100% of AMI': '#7E587E'
            , 'Greater than 100% of AMI': '#FBB117'
        }

        fig = px.bar(df_plot, x='Race Ethnicity', y='Percentage'
                     , color = columns
                     # , barmode='group'
                     , color_discrete_map=color_map)
        
        fig.update_traces(hovertemplate="%{y}")
        fig.update_yaxes(ticksuffix='%')
        fig.update_layout(legend={'traceorder': 'reversed'})

        path_plots = path_out / county / jurisdiction / 'Supplemental'
        plot_rhna(export=export)
    
        ## Exporting ---
        
        if export:
            
            export_rhna(df_prod, df_pct)
            

list_indicators.append(indicator_name)



***

FARM

***